# Paper 4: generative comparator only (second Colab run)

Open on an **L4** runtime and choose Runtime, then Run all. It mounts Drive, rebuilds and verifies the frozen inputs exactly as the first notebook did, then runs the open generative comparator (Qwen3-14B-AWQ through vLLM) on its eight conditions. Every output line is also written to `My Drive/Jev/paper4/colab/answers/comparator-open/comparator_log.txt`. The decision-model answers from the first run are not touched. Expected time is well under an hour; answers are written line by line, so a rerun resumes.

In [ ]:
import subprocess as _sp, sys as _sys
_sp.run([_sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, datetime

# Local (fast) storage: source, HF cache, regenerated inputs. Only answers live on Drive
# -- everything else is cheap to regenerate and would just slow every read/write down if
# it lived on the mounted filesystem instead.
LOCAL_ROOT = '/content/p4'
SHARED_LOCAL = f'{LOCAL_ROOT}/shared'      # mirrors the project's shared/ layout exactly
SRC_DIR = f'{SHARED_LOCAL}/src'            # so common.py's ROOT-relative INPUTS path,
DATA_DIR = f'{SHARED_LOCAL}/data'          # and harness.py's SHARED-relative default,
INPUTS_DIR = f'{SHARED_LOCAL}/colab/inputs'  # resolve to this same directory with no
                                            # env-var override needed for either.
os.environ['HF_HOME'] = '/content/hf'
os.environ['USE_TF'] = '0'

DRIVE_ROOT = '/content/drive/MyDrive/Jev/paper4/colab'
ANSWERS_DIR = f'{DRIVE_ROOT}/answers'
os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ANSWERS_DIR, exist_ok=True)
run_log = {'start': datetime.datetime.utcnow().isoformat() + 'Z', 'models': {}}
print('local root:', LOCAL_ROOT)
print('answers on Drive:', ANSWERS_DIR)

In [ ]:
%%bash
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
python3 -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)" 2>/dev/null || true

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
run_log['gpu'] = gpu_info
print(gpu_info)

## Write the source files

Embedded verbatim from the repository at build time (`build_notebook.py`).

In [ ]:
%%writefile /content/p4/shared/src/common.py
"""Shared paths, metrics and calibration helpers for the benchmark.

Adapted from paper3/src/common.py. Every analysis script imports this first,
because it sets the BLAS thread limits before numpy is imported.
"""
import os

for _v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

import json
import pickle

import numpy as np

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))   # paper4/shared/
BASE = os.path.dirname(os.path.dirname(ROOT))                         # project root
DATA = os.path.join(ROOT, "data")
INPUTS = os.path.join(ROOT, "colab", "inputs")
ANSWERS = os.path.join(ROOT, "answers")
RESULTS = os.path.join(ROOT, "results")
FIGURES = os.path.join(ROOT, "figures")
TABLES = os.path.join(ROOT, "tables")

MODEL = "jev-1.13.0"
PRICE_PER_MTOK = 0.042      # USD per million input tokens, docs.typesafe.ai, accessed 2026-09-24

SEED = 20260924


def _p(*parts):
    return os.path.join(BASE, *parts)


def ensure_dirs():
    for d in (RESULTS, FIGURES, TABLES):
        os.makedirs(d, exist_ok=True)


def dump(obj, name):
    """Write a result file incrementally so a crash does not cost the sweep."""
    ensure_dirs()
    path = os.path.join(RESULTS, name)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as fh:
        json.dump(obj, fh, indent=2, sort_keys=True, default=_jsonable)
    os.replace(tmp, path)
    return path


def _jsonable(o):
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(repr(o))


def normalize(P):
    P = np.clip(np.asarray(P, dtype=float), 1e-6, None)
    return P / P.sum(1, keepdims=True)


# ------------------------------------------------------------------ metrics

def nll(P, y):
    P = normalize(P)
    return float(-np.mean(np.log(P[np.arange(len(y)), y])))


def brier(P, y):
    P = normalize(P)
    Y = np.zeros_like(P)
    Y[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((P - Y) ** 2, axis=1)))


def accuracy(P, y):
    return float(np.mean(np.argmax(P, 1) == y))


def macro_f1(P, y, k):
    pred = np.argmax(P, 1)
    fs = []
    for c in range(k):
        tp = np.sum((pred == c) & (y == c))
        fp = np.sum((pred == c) & (y != c))
        fn = np.sum((pred != c) & (y == c))
        fs.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(fs))


def ece_equal_mass(P, y, bins=10):
    """Expected calibration error of the top-class probability, equal-mass bins."""
    P = normalize(P)
    conf = P.max(1)
    correct = (np.argmax(P, 1) == y).astype(float)
    order = np.argsort(conf)
    n = len(conf)
    edges = [int(round(i * n / bins)) for i in range(bins + 1)]
    tot = 0.0
    for a, b in zip(edges[:-1], edges[1:]):
        if b <= a:
            continue
        idx = order[a:b]
        tot += (b - a) / n * abs(conf[idx].mean() - correct[idx].mean())
    return float(tot)


def reliability(P, y, bins=10):
    """Equal-mass reliability curve of the top-class probability."""
    P = normalize(P)
    conf = P.max(1)
    correct = (np.argmax(P, 1) == y).astype(float)
    order = np.argsort(conf)
    n = len(conf)
    edges = [int(round(i * n / bins)) for i in range(bins + 1)]
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        if b <= a:
            continue
        idx = order[a:b]
        out.append({"n": int(b - a),
                    "mean_confidence": float(conf[idx].mean()),
                    "observed_accuracy": float(correct[idx].mean())})
    return out


def scores(P, y, k):
    return {"nll": nll(P, y), "brier": brier(P, y), "accuracy": accuracy(P, y),
            "macro_f1": macro_f1(P, y, k), "ece": ece_equal_mass(P, y)}


def bootstrap_nll(P, y, groups, reps=400, seed=SEED):
    """Cluster bootstrap over pedestrians, because rows within a pedestrian are dependent."""
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    index = {g: np.where(groups == g)[0] for g in uniq}
    vals = []
    for _ in range(reps):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([index[g] for g in pick])
        vals.append(nll(P[idx], y[idx]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)


# ------------------------------------------- temperature scaling and fitting

def apply_T(P, T):
    Q = np.exp(np.log(normalize(P)) / T)
    return Q / Q.sum(1, keepdims=True)


def fit_T(P, y, grid=None):
    grid = grid if grid is not None else np.exp(np.linspace(np.log(0.25), np.log(8.0), 60))
    best, bt = np.inf, 1.0
    for T in grid:
        v = nll(apply_T(P, T), y)
        if v < best:
            best, bt = v, float(T)
    return bt


def group_folds(groups, n_splits=5, seed=SEED):
    """Grouped folds balanced by size, shuffled so fold membership is not run order.

    sklearn's GroupKFold is deterministic and orders by group size, which for these
    datasets puts whole experimental runs together. Shuffling the group order first
    keeps folds disjoint by pedestrian while removing that structure.
    """
    rng = np.random.default_rng(seed)
    uniq = np.unique(groups)
    rng.shuffle(uniq)
    sizes = {g: int(np.sum(groups == g)) for g in uniq}
    loads = np.zeros(n_splits)
    assign = {}
    for g in sorted(uniq, key=lambda g: -sizes[g]):
        f = int(np.argmin(loads))
        assign[g] = f
        loads[f] += sizes[g]
    fold_of = np.array([assign[g] for g in groups])
    return [(np.where(fold_of != f)[0], np.where(fold_of == f)[0]) for f in range(n_splits)]


def cv_logit(X, y, k, folds, C=0.5, max_iter=3000, subsample=None, rng=None):
    """Out-of-fold probabilities from a multinomial logit on X.

    subsample, when set, limits each training fold to that many rows, which is how
    the learning curves are produced. Test folds are never subsampled.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    P = np.full((len(y), k), np.nan)
    for tr, te in folds:
        if subsample is not None and subsample < len(tr):
            tr = rng.choice(tr, size=subsample, replace=False)
        if len(np.unique(y[tr])) < 2:
            P[te] = np.bincount(y[tr], minlength=k) / len(tr)
            continue
        m = make_pipeline(StandardScaler(),
                          LogisticRegression(C=C, max_iter=max_iter))
        m.fit(X[tr], y[tr])
        pp = np.zeros((len(te), k))
        pp[:, m.classes_] = m.predict_proba(X[te])
        # classes absent from this training subset get the smoothed prior
        missing = [c for c in range(k) if c not in set(m.classes_)]
        if missing:
            pp = (pp + 1e-4) / (pp + 1e-4).sum(1, keepdims=True)
        P[te] = pp
    return normalize(P)


def cv_freq(y, k, folds):
    P = np.zeros((len(y), k))
    for tr, te in folds:
        P[te] = (np.bincount(y[tr], minlength=k) + 0.5) / (len(tr) + 0.5 * k)
    return normalize(P)


def cv_temp_scaled(P_raw, y, folds):
    """Temperature fitted inside the training folds only."""
    out = np.zeros_like(P_raw)
    for tr, te in folds:
        T = fit_T(P_raw[tr], y[tr])
        out[te] = apply_T(P_raw[te], T)
    return normalize(out)


In [ ]:
%%writefile /content/p4/shared/src/s00_freeze_inputs.py
"""Freeze every request the benchmark sends, one JSONL file per condition.

Every model, hosted or open, reads exactly these files, so the state, question and
option text are byte-identical across models. A record is

    {"key", "dataset", "condition", "item_id", "state", "questions", "gold", "meta"}

where "state" and "questions" together are the body of a POST /v1/systemone request,
and "gold" maps each question id to {"type", "label", "options", "soft"} with option
keys as they appear in that condition.

Conditions
    d1_native      typed-decisions test split, questions exactly as published
    d1_neutral     the same, Choice option keys replaced by o1..oK (rubrics unchanged)
    d1_calib       300 training-split states in the neutral form, for temperature fits
    e2_d1_<name>   every noul question of d1 recast as a two-option Choice whose keys
                   are 0/1, no/yes, yes/no (polarity swapped against the rubric) or
                   random strings; rubrics and their order are held fixed
    d2_k150        CLINC-150 test, 600 in-scope (4 per intent) + 200 out-of-scope
    d2_k50/k20/k5  the same 600 in-scope items with nested option subsets that
                   always contain the gold intent
    d2_hier_dom    domain Choice (10 options) for the 600 in-scope items
    d2_hier_int    intent Choice within the gold domain (15 options)
    d3_<task>      human-labelled social-science tasks, neutral keys
    e2_d3_<name>   binary d3 tasks under the four naming conditions

Run once. Re-running reproduces the same files, and the manifest records SHA-256.
"""
import hashlib
import json
import os
import random
import re
import string

import common as C

OUT = C.INPUTS
D1_REPO = "LocalLLaMA/typed-decisions"
NAMINGS = ["k01", "kny", "kswap", "krand"]


def neutral_keys(k):
    return [f"o{i + 1}" for i in range(k)]


def write(cond, rows, manifest):
    os.makedirs(OUT, exist_ok=True)
    path = os.path.join(OUT, f"{cond}.jsonl")
    with open(path, "w", encoding="utf-8", newline="\n") as fh:
        for r in rows:
            # no sort_keys: insertion order of each criteria dict is the option order
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    h = hashlib.sha256(open(path, "rb").read()).hexdigest()
    nq = sum(len(r["questions"]) for r in rows)
    manifest[cond] = {"file": f"{cond}.jsonl", "requests": len(rows), "decisions": nq,
                      "sha256": h}
    print(f"{cond:18s} {len(rows):5d} requests {nq:6d} decisions")


# ------------------------------------------------------------------ D1

def d1_rows(split, neutral, cond, pick=None):
    from datasets import load_dataset
    ds = load_dataset(D1_REPO, "all", split=split)
    rows = []
    for r in ds:
        if pick is not None and r["id"] not in pick:
            continue
        qs = json.loads(r["questions"])
        gold = json.loads(r["gold"])
        state = json.loads(r["state"])
        out_q, out_g = {}, {}
        for qid, q in qs.items():
            g = gold[qid]
            if q["type"] == "choice":
                names = list(q["criteria"].keys())
                keys = neutral_keys(len(names)) if neutral else names
                ren = dict(zip(names, keys))
                out_q[qid] = {"type": "choice", "instructions": q["instructions"],
                              "criteria": {ren[n]: q["criteria"][n] for n in names}}
                out_g[qid] = {"type": "choice", "label": ren[g["label"]], "options": keys,
                              "soft": {ren[n]: g["probabilities"][n] for n in names},
                              "native_options": names}
            elif q["type"] == "noul":
                out_q[qid] = q
                out_g[qid] = {"type": "noul", "label": g["label"], "options": ["false", "true"],
                              "soft": {"false": g["probabilities"]["false"],
                                       "true": g["probabilities"]["true"]}}
            else:
                lv = [str(i) for i in range(len(q["criteria"]))]
                out_q[qid] = q
                out_g[qid] = {"type": "score", "label": g["label"], "options": lv,
                              "soft": {k: g["probabilities"][k] for k in lv}}
        rows.append({"key": f"{cond}|{r['id']}", "dataset": "d1", "condition": cond,
                     "item_id": r["id"], "state": state, "questions": out_q, "gold": out_g,
                     "meta": {"workflow": r["workflow"],
                              "label_agreement": json.loads(r["label_agreement"])}})
    return rows


def naming_pair(name, rng):
    """Keys for (negative rubric, positive rubric) under one naming condition."""
    if name == "k01":
        return "0", "1"
    if name == "kny":
        return "no", "yes"
    if name == "kswap":
        return "yes", "no"
    a = "".join(rng.choice(string.ascii_lowercase) for _ in range(5))
    b = a
    while b == a:
        b = "".join(rng.choice(string.ascii_lowercase) for _ in range(5))
    return a, b


def e2_from_binary(base_rows, name, cond, binary_fn):
    """Recast binary questions under a naming condition, rubric order held fixed.

    binary_fn(row, qid) returns (instructions, neg_rubric, pos_rubric, gold_is_pos,
    soft_pos) for a binary question, or None to skip it.
    """
    rng = random.Random(f"{C.SEED}-{cond}")
    rows = []
    for r in base_rows:
        out_q, out_g = {}, {}
        for qid in r["questions"]:
            b = binary_fn(r, qid)
            if b is None:
                continue
            instr, neg, pos, is_pos, soft_pos = b
            kn, kp = naming_pair(name, rng)
            out_q[qid] = {"type": "choice", "instructions": instr,
                          "criteria": {kn: neg, kp: pos}}
            out_g[qid] = {"type": "choice", "label": kp if is_pos else kn,
                          "options": [kn, kp], "positive": kp,
                          "soft": None if soft_pos is None else {kn: 1 - soft_pos, kp: soft_pos}}
        if out_q:
            rows.append({"key": f"{cond}|{r['item_id']}", "dataset": r["dataset"],
                         "condition": cond, "item_id": r["item_id"], "state": r["state"],
                         "questions": out_q, "gold": out_g,
                         "meta": dict(r["meta"], naming=name)})
    return rows


def d1_binary(r, qid):
    q, g = r["questions"][qid], r["gold"][qid]
    if q["type"] != "noul":
        return None
    crit = q.get("criteria") or {"false": "No.", "true": "Yes."}
    return (q["instructions"], crit["false"], crit["true"], g["label"] == "true",
            g["soft"]["true"])


# ------------------------------------------------------------------ D2

def human(s):
    return s.replace("_", " ")


def d2_all(manifest):
    d = json.load(open(os.path.join(C.DATA, "d2", "data_full.json"), encoding="utf-8"))
    domains = json.load(open(os.path.join(C.DATA, "d2", "domains.json"), encoding="utf-8"))
    dom_of = {i: dname for dname, ints in domains.items() for i in ints}
    intents = sorted(dom_of)
    assert len(intents) == 150
    rng = random.Random(C.SEED)
    by_int = {}
    for text, lab in d["test"]:
        by_int.setdefault(lab, []).append(text)
    ins = []
    for lab in intents:
        pick = rng.sample(range(len(by_int[lab])), 4)
        ins += [(f"{lab}_{i:02d}", by_int[lab][i], lab) for i in sorted(pick)]
    oos_texts = [t for t, _ in d["oos_test"]]
    oos_idx = sorted(rng.sample(range(len(oos_texts)), 200))
    oos = [(f"oos_{i:04d}", oos_texts[i], None) for i in oos_idx]

    q_int = "Which intent does this request to a virtual assistant express?"
    q_dom = "Which domain does this request to a virtual assistant belong to?"

    def choice_row(cond, iid, text, options, gold_name, instr, meta):
        keys = neutral_keys(len(options))
        ren = dict(zip(options, keys))
        return {"key": f"{cond}|{iid}", "dataset": "d2", "condition": cond, "item_id": iid,
                "state": text,
                "questions": {"intent": {"type": "choice", "instructions": instr,
                                         "criteria": {ren[o]: human(o) for o in options}}},
                "gold": {"intent": {"type": "choice",
                                    "label": ren[gold_name] if gold_name else None,
                                    "options": keys, "native_options": options,
                                    "soft": None}},
                "meta": meta}

    # one fixed permutation of the 150 intents per item, and nested subsets
    orders = {}
    for iid, text, lab in ins + oos:
        perm = intents[:]
        rng.shuffle(perm)
        orders[iid] = perm
    rows150 = [choice_row("d2_k150", iid, text, orders[iid], lab, q_int,
                          {"in_scope": lab is not None, "intent": lab,
                           "domain": dom_of.get(lab)})
               for iid, text, lab in ins + oos]
    write("d2_k150", rows150, manifest)
    for k in (50, 20, 5):
        rows = []
        for iid, text, lab in ins:
            others = [o for o in orders[iid] if o != lab]
            keep = set(others[:k - 1]) | {lab}          # nested: prefix of one permutation
            opts = [o for o in orders[iid] if o in keep]
            rows.append(choice_row(f"d2_k{k}", iid, text, opts, lab, q_int,
                                   {"in_scope": True, "intent": lab, "domain": dom_of[lab]}))
        write(f"d2_k{k}", rows, manifest)
    dnames = sorted(domains)
    rows_dom, rows_int = [], []
    for iid, text, lab in ins:
        dperm = dnames[:]
        rng.shuffle(dperm)
        r = choice_row("d2_hier_dom", iid, text, dperm, dom_of[lab], q_dom,
                       {"in_scope": True, "intent": lab, "domain": dom_of[lab]})
        r["questions"]["domain"] = r["questions"].pop("intent")
        r["gold"]["domain"] = r["gold"].pop("intent")
        rows_dom.append(r)
        iperm = [o for o in orders[iid] if dom_of[o] == dom_of[lab]]
        rows_int.append(choice_row("d2_hier_int", iid, text, iperm, lab, q_int,
                                   {"in_scope": True, "intent": lab, "domain": dom_of[lab]}))
    write("d2_hier_dom", rows_dom, manifest)
    write("d2_hier_int", rows_int, manifest)


# ------------------------------------------------------------------ D3

def d3_all(manifest):
    ddir = os.path.join(C.DATA, "d3")
    if not os.path.isdir(ddir):
        print("D3 not prepared yet; skipped")
        return
    for fn in sorted(os.listdir(ddir)):
        if not fn.endswith(".jsonl"):
            continue
        task = fn[:-6]
        items = [json.loads(l) for l in open(os.path.join(ddir, fn), encoding="utf-8")]
        cond = f"d3_{task}"
        rows = []
        for it in items:
            opts = it["options"]
            keys = neutral_keys(len(opts))
            ren = dict(zip(opts, keys))
            rows.append({"key": f"{cond}|{it['id']}", "dataset": "d3", "condition": cond,
                         "item_id": str(it["id"]), "state": it["text"],
                         "questions": {task: {"type": "choice",
                                              "instructions": it["question"],
                                              "criteria": {ren[o]: it["option_desc"][o]
                                                           for o in opts}}},
                         "gold": {task: {"type": "choice", "label": ren[it["gold"]],
                                         "options": keys, "native_options": opts,
                                         "soft": None}},
                         "meta": {"task": task, "binary": bool(it["binary"]),
                                  "positive": it.get("positive"),
                                  "native_gold": it["gold"]}})
        write(cond, rows, manifest)
        if items[0]["binary"]:
            def fn_bin(r, qid, _items={str(i["id"]): i for i in items}):
                it = _items[r["item_id"]]
                pos = it["positive"]
                neg = [o for o in it["options"] if o != pos][0]
                # the package's binary rubrics open with "True:" or "False:"; that
                # prefix would carry polarity independently of the option name, so the
                # naming conditions strip it and keep the rest of the rubric verbatim
                strip = lambda s: re.sub(r"^(True|False):\s*", "", s)
                return (it["question"], strip(it["option_desc"][neg]),
                        strip(it["option_desc"][pos]), it["gold"] == pos, None)
            for name in NAMINGS:
                write(f"e2_d3_{task}_{name}",
                      e2_from_binary(rows, name, f"e2_d3_{task}_{name}", fn_bin), manifest)


def main():
    os.environ.setdefault("HF_HOME", "D:/p4env/hf")
    manifest = {}
    native = d1_rows("test", False, "d1_native")
    write("d1_native", native, manifest)
    neutral = d1_rows("test", True, "d1_neutral")
    write("d1_neutral", neutral, manifest)
    from datasets import load_dataset
    tr = load_dataset(D1_REPO, "all", split="train")
    rng = random.Random(C.SEED)
    pick = set()
    for wf in sorted(set(tr["workflow"])):
        ids = sorted(r["id"] for r in tr if r["workflow"] == wf)
        pick |= set(rng.sample(ids, 75))
    write("d1_calib", d1_rows("train", True, "d1_calib", pick), manifest)
    for name in NAMINGS:
        write(f"e2_d1_{name}", e2_from_binary(native, name, f"e2_d1_{name}", d1_binary),
              manifest)
    d2_all(manifest)
    d3_all(manifest)
    # the test-retest subset: 10 states per workflow of d1_neutral, 200 decisions
    rng = random.Random(f"{C.SEED}-retest")
    sub = []
    for wf in sorted({r["meta"]["workflow"] for r in neutral}):
        ids = sorted(r["item_id"] for r in neutral if r["meta"]["workflow"] == wf)
        sub += rng.sample(ids, 10)
    manifest["_retest_subset"] = {"condition": "d1_neutral", "item_ids": sorted(sub)}
    manifest["_seed"] = C.SEED
    manifest["_sources"] = {
        "d1": f"https://huggingface.co/datasets/{D1_REPO} revision c76749ec58bd8c3d2ea706b31c333a9059c38f90",
        "d2": "https://github.com/clinc/oos-eval data/data_full.json and domains.json, commit "
              + open(os.path.join(C.DATA, "d2", "CLINC_COMMIT")).read().strip()}
    with open(os.path.join(OUT, "manifest.json"), "w", encoding="utf-8") as fh:
        json.dump(manifest, fh, indent=1, sort_keys=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/p4/shared/src/harness.py
"""One harness for every decision model in the benchmark.

A backend answers one frozen request, which is the body of a POST /v1/systemone call
({"state", "questions"}), and returns a dict of normalized answers

    {qid: {"probs": {option_key: p, ...}, "raw": <the model's own answer object>}}

where option keys are those of the request ("false"/"true" for noul, "0".."L-1" for
score). The runner writes one append-only JSONL line per request, keyed by the
record key, with the model identifier, the revision, the repeat index, the run date,
the wall-clock latency and the input-token usage when the backend reports it. An
interrupted run resumes without repeating work.

Backends
    jev        TypeSafe API through typesafe-sdk, model pinned to jev-1.13.0
    systemone  any server that implements the same wire format (laya-serve,
               kev.serve, decider.serve), through plain HTTP
    others     in-process adapters for the open models live in adapters.py and
               register themselves through register()

Usage
    python harness.py --model jev --cond d1_neutral --rep 1 [--limit 10]
    python harness.py --model jev --cond all --rep 1 --stage estimate
"""
import argparse
import asyncio
import datetime as dt
import hashlib
import json
import os
import sys
import time

HERE = os.path.dirname(os.path.abspath(__file__))
SHARED = os.path.dirname(HERE)
INPUTS = os.environ.get("P4_INPUTS", os.path.join(SHARED, "colab", "inputs"))
ANSWERS = os.environ.get("P4_ANSWERS", os.path.join(SHARED, "answers"))
PRICE_PER_MTOK = 0.042          # USD per million input tokens for Jev, accessed 2026-09-24
SPEND_TRIPWIRE = float(os.environ.get("P4_SPEND_TRIPWIRE", "20.0"))

BACKENDS = {}


def register(name, factory):
    BACKENDS[name] = factory


# ------------------------------------------------------------------ io

def load_inputs(cond):
    path = os.path.join(INPUTS, f"{cond}.jsonl")
    with open(path, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]


def all_conditions():
    man = json.load(open(os.path.join(INPUTS, "manifest.json"), encoding="utf-8"))
    return [c for c in man if not c.startswith("_")]


def out_path(model_tag, cond, rep):
    d = os.path.join(ANSWERS, model_tag)
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, f"{cond}__rep{rep}.jsonl")


def done_keys(path):
    keys = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    j = json.loads(line)
                    if not j.get("error"):
                        keys.add(j["key"])
                except (json.JSONDecodeError, KeyError):
                    continue
    return keys


def request_hash(rec):
    body = json.dumps({"state": rec["state"], "questions": rec["questions"]},
                      ensure_ascii=False)
    return hashlib.sha256(body.encode("utf-8")).hexdigest()[:16]


# ------------------------------------------------------------------ normalization

def normalize_answer(q, a):
    """Map one answer object in the System One response shape to option probabilities."""
    t = q["type"]
    if t == "noul":
        p = a.get("noul")
        if p is None and "probabilities" in a:
            pr = a["probabilities"]
            p = pr.get("true", pr.get("yes"))
        p = float(p)
        return {"false": 1.0 - p, "true": p}
    if t == "score":
        pr = a["probabilities"]
        return {str(k): float(v) for k, v in pr.items()}
    pr = a["probabilities"]
    return {str(k): float(v) for k, v in pr.items()}


def normalize_response(questions, answers):
    out = {}
    for qid, q in questions.items():
        a = answers.get(qid)
        if a is None:
            out[qid] = {"probs": None, "raw": None}
            continue
        out[qid] = {"probs": normalize_answer(q, a), "raw": a}
    return out


# ------------------------------------------------------------------ backends

class JevBackend:
    """TypeSafe API. The alias jev-latest moves, so the version is always pinned."""
    tag = "jev-1.13.0"
    revision = "jev-1.13.0"
    served_temperature = None
    concurrency = 10

    def __init__(self, model="jev-1.13.0"):
        from dotenv import load_dotenv
        load_dotenv(os.path.join(os.path.dirname(os.path.dirname(SHARED)), ".env"))
        self.model = model
        self.tag = model
        self.revision = model

    async def __aenter__(self):
        from typesafe_sdk import AsyncTypeSafeClient
        self.client = AsyncTypeSafeClient(timeout=90)
        await self.client.__aenter__()
        return self

    async def __aexit__(self, *a):
        await self.client.__aexit__(*a)

    async def answer(self, rec):
        r = await self.client.system_one(state=rec["state"], questions=rec["questions"],
                                         model=self.model)
        d = r.model_dump() if hasattr(r, "model_dump") else dict(r)
        answers = d["answers"]
        answers = {k: (v if isinstance(v, dict) else dict(v)) for k, v in answers.items()}
        return {"model_returned": d.get("model"),
                "usage_in": d["usage"]["input_tokens"] if d.get("usage") else None,
                "answers": normalize_response(rec["questions"], answers)}


class SystemOneHTTP:
    """Any local server that implements POST /v1/systemone (laya-serve, kev.serve,
    decider.serve). The caller supplies the tag, revision and served temperature."""
    concurrency = 1

    def __init__(self, base_url, tag, revision, served_temperature=None, model=None,
                 concurrency=1):
        self.base_url = base_url.rstrip("/")
        self.tag, self.revision = tag, revision
        self.served_temperature = served_temperature
        self.model = model
        self.concurrency = concurrency

    async def __aenter__(self):
        import httpx
        self.client = httpx.AsyncClient(timeout=300)
        return self

    async def __aexit__(self, *a):
        await self.client.aclose()

    async def answer(self, rec):
        body = {"state": rec["state"], "questions": rec["questions"]}
        if self.model:
            body["model"] = self.model
        r = await self.client.post(f"{self.base_url}/v1/systemone", json=body)
        r.raise_for_status()
        d = r.json()
        return {"model_returned": d.get("model"),
                "usage_in": (d.get("usage") or {}).get("input_tokens"),
                "answers": normalize_response(rec["questions"], d["answers"])}


register("jev", lambda **kw: JevBackend(**kw))


# ------------------------------------------------------------------ runner

async def run(backend, cond, rep, limit=None, only_keys=None):
    recs = load_inputs(cond)
    if only_keys is not None:
        recs = [r for r in recs if r["item_id"] in only_keys]
    if limit:
        recs = recs[:limit]
    path = out_path(backend.tag, cond, rep)
    have = done_keys(path)
    todo = [r for r in recs if r["key"] not in have]
    print(f"[{backend.tag} {cond} rep{rep}] {len(todo)} to run, {len(have)} cached",
          flush=True)
    if not todo:
        return path
    sem = asyncio.Semaphore(getattr(backend, "concurrency", 1))
    lock = asyncio.Lock()
    stats = {"n": 0, "err": 0, "tok": 0}
    t0 = time.time()
    run_date = dt.date.today().isoformat()
    fh = open(path, "a", encoding="utf-8")

    async def one(rec):
        async with sem:
            last = None
            for attempt in range(7):
                try:
                    st = time.perf_counter()
                    out = await backend.answer(rec)
                    lat = time.perf_counter() - st
                    line = {"key": rec["key"], "condition": cond, "rep": rep,
                            "model": backend.tag, "revision": backend.revision,
                            "model_returned": out.get("model_returned"),
                            "served_temperature": backend.served_temperature,
                            "run_date": run_date, "latency_s": round(lat, 4),
                            "usage_in": out.get("usage_in"),
                            "request_hash": request_hash(rec),
                            "answers": out["answers"]}
                    async with lock:
                        fh.write(json.dumps(line, ensure_ascii=False) + "\n")
                        # HOOK (adapters.py, documented in its module docstring): flush every
                        # line, not every 200. The Colab notebook writes answers straight to a
                        # mounted Drive folder so a disconnect costs nothing; that only holds if
                        # each line is actually on disk before the next request starts.
                        fh.flush()
                        stats["n"] += 1
                        stats["tok"] += out.get("usage_in") or 0
                        if stats["n"] % 200 == 0:
                            cost = stats["tok"] * PRICE_PER_MTOK / 1e6
                            print(f"  {stats['n']}/{len(todo)} {time.time() - t0:.0f}s "
                                  f"tokens {stats['tok']} ${cost:.3f}", flush=True)
                    return
                except Exception as e:          # noqa: BLE001, retried or recorded
                    last = repr(e)
                    if any(t in last for t in ("429", "529", "502", "503", "Timeout",
                                               "Connect", "RemoteProtocol")):
                        await asyncio.sleep(min(2 ** attempt, 30))
                        continue
                    break
            async with lock:
                stats["err"] += 1
                fh.write(json.dumps({"key": rec["key"], "condition": cond, "rep": rep,
                                     "model": backend.tag, "error": last[:500]}) + "\n")
                fh.flush()   # HOOK (adapters.py): same every-line-flush reasoning as the ok path
                if stats["err"] <= 3:
                    print(f"  ERR {rec['key']}: {last[:200]}", flush=True)

    async with backend:
        # chunks keep the spend tripwire effective within one condition
        for i in range(0, len(todo), 500):
            chunk = todo[i:i + 500]
            # HOOK (adapters.py, documented in its module docstring): a backend that wants to
            # batch a whole chunk through one call (comparator-open's vLLM offline LLM.generate)
            # may define an async prefetch(records) that fills its own cache; answer(rec) then
            # just looks the result up. Backends without prefetch are unaffected.
            prefetch = getattr(backend, "prefetch", None)
            if prefetch is not None:
                await prefetch(chunk)
            await asyncio.gather(*(one(r) for r in chunk))
            spent = spend_so_far()
            if spent > SPEND_TRIPWIRE:
                print(f"SPEND TRIPWIRE: ${spent:.2f} > ${SPEND_TRIPWIRE}", flush=True)
                break
    fh.close()
    cost = stats["tok"] * PRICE_PER_MTOK / 1e6
    print(f"[{backend.tag} {cond} rep{rep}] done {stats['n']} ok, {stats['err']} errors, "
          f"{time.time() - t0:.0f}s, tokens {stats['tok']} (${cost:.3f} at Jev price)",
          flush=True)
    return path


def spend_so_far():
    """Jev spend across every Jev answer file, from recorded usage."""
    tok = 0
    d = os.path.join(ANSWERS)
    if not os.path.isdir(d):
        return 0.0
    for tag in os.listdir(d):
        if not tag.startswith("jev"):
            continue
        for fn in os.listdir(os.path.join(d, tag)):
            with open(os.path.join(d, tag, fn), encoding="utf-8") as fh:
                for line in fh:
                    try:
                        tok += json.loads(line).get("usage_in") or 0
                    except json.JSONDecodeError:
                        pass
    return tok * PRICE_PER_MTOK / 1e6


def estimate(conds, reps):
    """Rough Jev cost before anything is sent: characters / 3.5 as tokens."""
    tot_tok = 0
    for c in conds:
        for r in load_inputs(c):
            body = json.dumps({"state": r["state"], "questions": r["questions"]})
            tot_tok += len(body) / 3.5
    print(f"{len(conds)} conditions x {reps} repeats ~ {tot_tok * reps / 1e6:.2f} M tokens"
          f" ~ ${tot_tok * reps * PRICE_PER_MTOK / 1e6:.2f} at the Jev price")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--cond", required=True, help="condition name, comma list, or all")
    ap.add_argument("--rep", type=int, default=1)
    ap.add_argument("--limit", type=int)
    ap.add_argument("--retest", action="store_true",
                    help="only the 40-state test-retest subset of d1_neutral")
    ap.add_argument("--stage", default="run", choices=["run", "estimate"])
    a = ap.parse_args()
    conds = all_conditions() if a.cond == "all" else a.cond.split(",")
    if a.stage == "estimate":
        estimate(conds, a.rep)
        return
    only = None
    if a.retest:
        man = json.load(open(os.path.join(INPUTS, "manifest.json"), encoding="utf-8"))
        only = set(man["_retest_subset"]["item_ids"])
    try:
        import adapters  # noqa: F401  registers the open-model backends
    except ImportError:
        pass
    backend = BACKENDS[a.model]()
    for c in conds:
        asyncio.run(run(backend, c, a.rep, a.limit, only))


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/p4/shared/src/adapters.py
"""In-process (and one subprocess-free, one HTTP-served) backends for the open decision models.

Registers with harness.register() so `python harness.py --model <tag> --cond ...` works exactly
like the Jev backend. Import this module lazily (harness.main() does `import adapters` inside a
try/except ImportError) -- so THIS FILE MUST NOT IMPORT TORCH, TRANSFORMERS, LAYA, KEV, DECIDER,
THISTHAT OR VLLM AT MODULE SCOPE. Every backend below defers those imports to __aenter__ (or to a
worker thread/process invoked from __aenter__), so `import adapters` succeeds in any uv env
regardless of which of these packages that env actually has installed; only asking harness.py to
run a *specific* --model requires that model's dependency to be present.

Documented hooks this module relies on in harness.py (see harness.py's run(); both are backward
compatible -- a backend that does not use them behaves exactly as it did before this file existed):

  1. Every JSONL line is flushed to disk immediately, not every 200. The Colab notebook writes
     answers straight into a mounted Drive folder specifically so a disconnect costs nothing; that
     guarantee only holds if a line is actually on disk before the next request starts.
  2. Before each up-to-500-record chunk, run() calls `await backend.prefetch(chunk)` if the
     backend defines `prefetch`. comparator-open's backend uses this to run one vLLM
     `LLM.generate()` call over the whole chunk (`--stage`/batched inference) instead of one
     `generate()` per request; `answer()` then just looks its record up in the cache prefetch
     filled. Backends without `prefetch` (everything except comparator-open) are unaffected.

Precision convention. Every backend returns, per question id,

    {"probs": {option_key: p, ...}, "raw": {...}}

"probs" is always the model's *as-served* distribution -- the temperature it ships with, applied
the way its own code applies it -- at full float precision. Several of these projects round their
own wire-format probabilities to 4 decimal places before returning them (Laya's `agent.predict`,
decider's `system_one`, Kev's `kev.api.to_answers` / `kev.serve` all do this, `kev/api.py`'s
`round_prob` says so explicitly: "4 decimals keeps the sum ... within TypeSafe's tolerance"). This
module routes around that rounding case by case (documented in each backend below) rather than
serializing through the model's own wire format, because the task requires full float precision
with no rounding.

"raw" always carries "served_temperature" and "probs_t1" (the T=1, i.e. un-tempered, distribution)
at the same full precision, plus whatever native fields the model's own answer object provides.
Where a second forward pass is not needed to get both distributions, this module does not run one:
softmax is scale invariant to an additive shift, so if p_T = softmax(z / T) is known at full
precision for one T, the distribution at any other temperature T' is recoverable exactly (not
approximately) as

    p_T' = normalize(p_T ** (T / T'))

(derivation: p_T,i = exp(z_i/T) / sum_j exp(z_j/T); write exp(z_i) = p_1,i * S with S = sum_j
exp(z_j) constant over i; then exp(z_i/T) = p_1,i^(1/T) * S^(1/T), and S^(1/T) cancels in the
softmax's normalization, giving p_T ∝ p_1^(1/T), i.e. p_1 = normalize(p_T ** T); the general form
follows by applying this twice). `_pow_normalize` below implements this. It is used for Laya
(recovering pre-bucket-temperature probabilities), Kev (recovering the served, at-T probabilities
from a single T=1 forward pass) and decider (recovering T=1 choice/noul probabilities, and, per
isolated Score level, the T=1 "does this level fit" probability before recombining levels).

Idempotent loading. harness.main() builds one backend instance per --model invocation, but
harness.run() is called once per --cond entry (once per condition; `--cond all` means once per
condition in manifest.json, i.e. ~24 times) and every call does `async with backend:`. A naive
__aenter__ would therefore reload a checkpoint once per condition. Every backend below loads once
(a `self._loaded` guard) and __aexit__ is a no-op; the process exits (or the notebook cell moves
to the next model, in its own subprocess/uv env) to free the GPU.
"""
import asyncio
import json
import math
import os
import sys

# harness.py's own CLI runs it as __main__ and does `import adapters` from inside main(). A plain
# `import harness` here would then import harness.py a SECOND time under the module name
# "harness", re-executing its top level and creating an independent BACKENDS = {} dict that our
# register() calls below would fill -- while main() keeps reading its own __main__-scoped
# BACKENDS, which would then only ever contain "jev". (Verified: running `python harness.py
# --model laya-en ...` raised KeyError('laya-en') from BACKENDS[a.model] for exactly this reason.)
# Reusing whichever module object is already the running harness.py -- sys.modules["harness"] if
# it was imported normally, sys.modules["__main__"] if harness.py itself is the running script --
# avoids the double import and its split-BACKENDS bug without changing harness.py. The __main__
# check requires BACKENDS/register to already be defined there (true once main() reaches its
# `import adapters` line, since that is after harness.py's whole module body has run) so that
# importing adapters.py from an unrelated script (test_adapters.py, a notebook cell) still falls
# through to a plain, correct `import harness` instead of mistaking that script's own __main__ for
# harness.py.
_main = sys.modules.get("__main__")
if "harness" in sys.modules:
    harness = sys.modules["harness"]
elif _main is not None and hasattr(_main, "BACKENDS") and hasattr(_main, "register"):
    harness = _main
else:
    import harness

# ---------------------------------------------------------------------- shared helpers


def _txt(v):
    """Render an option/instructions value that may be a string or any JSON value, the same
    convention every one of these wire formats uses (decider's systemone._txt, kev's api.render)."""
    return v if isinstance(v, str) else json.dumps(v, ensure_ascii=False)


def _state_text(state):
    """A JSON state rendered compactly; a string state passed through unchanged (this-that and
    Nimble need a single context string; Laya, Kev and decider accept the raw JSON state)."""
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def _pow_normalize(p, exponent):
    """{key: p ** exponent}, renormalized -- see the module docstring for the exact identity this
    implements. `p` values are cast to float first; a genuine 0 stays 0 for any positive
    exponent."""
    ex = {k: (float(v) ** exponent if v > 0 else 0.0) for k, v in p.items()}
    s = sum(ex.values())
    if s <= 0:
        n = len(ex) or 1
        return {k: 1.0 / n for k in ex}
    return {k: v / s for k, v in ex.items()}


def wire_keys(qtype, criteria):
    """The option keys a question's probabilities are reported under, in option order: the
    criteria names (choice), ["false", "true"] (noul), the level indices as strings (score). Pure
    function (no model import) so test_adapters.py can check it directly; mirrors kev/api.py's
    `question_keys` and decider/systemone.py's `render_question` name lists, which every adapter
    below must agree with since the freeze files define option order by dict/list insertion order.
    """
    if qtype == "choice":
        return list(criteria)
    if qtype == "noul":
        return ["false", "true"]
    if qtype == "score":
        return [str(i) for i in range(len(criteria))]
    raise ValueError(f"unknown question type {qtype!r}")


def noul_criteria(q):
    """{"false": ..., "true": ...}, defaulting to plain "No."/"Yes." the way harness.py's
    JevBackend and every one of these model cards do when a noul question ships no criteria."""
    c = q.get("criteria") or {}
    return {"false": c.get("false", "No."), "true": c.get("true", "Yes.")}


async def _run_sync(fn, *a, **kw):
    """Run a blocking (CPU/GPU) call off the event loop. Every backend's forward pass is
    synchronous torch code; concurrency=1 on all of them (one loaded model, one GPU) so this just
    keeps the loop responsive between requests, not real parallelism."""
    loop = asyncio.get_event_loop()
    if kw:
        import functools
        fn = functools.partial(fn, **kw)
    return await loop.run_in_executor(None, fn, *a)


def _model_sha(repo, revision=None):
    import huggingface_hub as hh
    return hh.model_info(repo, revision=revision).sha


# ---------------------------------------------------------------------- Laya (english / multilingual)

LAYA_REPO = "convaiinnovations/laya"


class LayaBackend:
    """convaiinnovations/laya, root checkpoint (English) or subfolder="multilingual".

    laya.load()/laya.agent.Agent() take no `revision` kwarg (verified against laya 0.3.20's
    source: Agent.__init__ calls `snapshot_download(model_id_or_path, **kw)` with no revision in
    `kw`). To pin the checkpoint anyway (the task requires it, and "not the Router" -- we use
    laya.load()'s single-checkpoint path, never laya.Router), we pre-resolve the pinned commit's
    snapshot ourselves with huggingface_hub.snapshot_download(repo, revision=sha, ...) and hand
    Agent() the resulting local directory; Agent() only downloads when its first argument is not
    an existing path, so a local directory is used exactly as given (agent.py's `if subfolder:
    model_dir = os.path.join(model_dir, subfolder)` still runs, so subfolder selection works the
    same as with a hub id).

    Precision: laya/agent.py's `_decode_answers` rounds every probability, score and confidence to
    4 decimal places with a bare `round(...)` call (agent.py lines ~675-703) before returning it.
    That name resolves at call time against the *module's* globals (LEGB), so assigning
    `laya.agent.round = <identity>` once, before the first predict(), makes every later call in
    that module skip rounding -- the temperature-bucket lookup, masking and softmax it wraps are
    completely untouched, only the final `round(x, 4)` becomes `x`. This is verified in the smoke
    test: the unrounded probabilities agree with `agent.predict()`'s normal (rounded) output to
    within 5e-5 on the same input.

    Laya ships one temperature per (question type, option-count) bucket rather than one scalar
    (`agent.temperature_by_options`, falling back to `agent.temperature[qtype]`; see
    laya/common.py's `temp_bucket`). `served_temperature` on the backend is therefore the
    sentinel 1.0 the task allows ("the temperature the model applies by default, or 1.0"); the
    exact bucket temperature actually used for each question is in that answer's
    raw["served_temperature_bucket"], and the whole map is in raw["temperature_buckets"].
    `head_max_len` and `max_len` (the option/state token budget the model card calls out as the
    source of high-K truncation) are recorded in raw on every answer, as the task asks.
    """

    concurrency = 1

    def __init__(self, subfolder=None, tag=None):
        self.subfolder = subfolder
        self.tag = tag or ("laya-ml" if subfolder else "laya-en")
        self._loaded = False

    async def __aenter__(self):
        if not self._loaded:
            await _run_sync(self._load)
            self._loaded = True
        return self

    async def __aexit__(self, *a):
        return None

    def _load(self):
        os.environ.setdefault("USE_TF", "0")
        import huggingface_hub as hh

        self.revision = hh.model_info(LAYA_REPO).sha
        prefix = f"{self.subfolder}/" if self.subfolder else ""
        local_dir = hh.snapshot_download(
            LAYA_REPO, revision=self.revision,
            allow_patterns=[prefix + p for p in
                            ("rl_agent_config.json", "model.safetensors", "tokenizer/*", "encoder/*")])

        import laya
        import laya.agent as _agent_mod
        if not getattr(_agent_mod, "_p4_unrounded", False):
            _agent_mod.round = lambda x, ndigits=None: x   # see class docstring
            _agent_mod._p4_unrounded = True
        from laya.common import QTYPES, temp_bucket
        self._QTYPES, self._temp_bucket = QTYPES, temp_bucket

        self.agent = laya.load(local_dir, subfolder=self.subfolder)
        self.served_temperature = 1.0
        self._temp_map = {"by_option_count": dict(self.agent.temperature_by_options),
                          "by_type": list(self.agent.temperature)}
        self._cfg = {"head_max_len": self.agent.cfg.get("head_max_len"),
                    "max_len": self.agent.cfg.get("max_len")}

    def _t_scale(self, qtype, k):
        bucket = self._temp_bucket(self._QTYPES[qtype], k)
        return self._temp_map["by_option_count"].get(bucket, self._temp_map["by_type"][self._QTYPES[qtype]])

    async def answer(self, rec):
        result = await _run_sync(self.agent.predict, rec["state"], rec["questions"])
        answers = harness.normalize_response(rec["questions"], result["answers"])
        out = {}
        for qid, a in answers.items():
            q = rec["questions"][qid]
            k = len(q["criteria"]) if q["type"] != "noul" else 2
            t = self._t_scale(q["type"], k)
            native = result["answers"][qid]
            raw = {"served_temperature_bucket": t, "temperature_buckets": self._temp_map,
                  "head_max_len": self._cfg["head_max_len"], "max_len": self._cfg["max_len"],
                  "native": native}
            probs = a["probs"] if a["probs"] is not None else None
            raw["probs_t1"] = _pow_normalize(probs, t) if probs is not None else None
            out[qid] = {"probs": probs, "raw": raw}
        return {"model_returned": result.get("model"),
               "usage_in": (result.get("usage") or {}).get("input_tokens"),
               "answers": out}


harness.register("laya-en", lambda **kw: LayaBackend(subfolder=None, tag="laya-en", **kw))
harness.register("laya-ml", lambda **kw: LayaBackend(subfolder="multilingual", tag="laya-ml", **kw))


# ---------------------------------------------------------------------- Kev (0.8b / 9b)

KEV_REPOS = {"kev-0.8b": "jaredpalmer/kev-0.8b", "kev-9b": "jaredpalmer/kev-9b"}
KEV_GITHUB_COMMIT = "73504e51f6ce2ade19c7819d4a5f2d84363cd40f"   # jaredpalmer/kev@main, resolved 2026-09-24


class KevBackend:
    """jaredpalmer/kev-0.8b or -9b: a LoRA + pointer head on Qwen3.5, served at a built-in
    temperature read from the checkpoint itself (kev.checkpoint.Meta.temperature, head.pt), never
    hard-coded here: the model card's published figures (T=2.41 for 0.8B, 2.30 for 9B) are a
    snapshot, and the actual served value can move with the checkpoint -- confirmed on this
    laptop, where the current kev-0.8b weights read T=2.3510958125672174, not the card's 2.41.
    `self.served_temperature = float(self.model.head.temperature)` below is always the live value.

    Uses kev's in-process loader (kev.checkpoint.Checkpoint, per the task: "use its in-process
    loader if one exists") rather than starting `python -m kev.serve` as a subprocess: kev.serve's
    /v1/systemone response goes through kev.api.to_answers -> round_prob, which rounds every
    probability to 4 decimals (kev/api.py: "Serialization precision ... 4 decimals keeps the sum
    ... within TypeSafe's tolerance"), so the HTTP path cannot give full precision no matter what
    temperature it is asked to serve at.

    Precision and both temperatures from one forward pass. `PointerHead.forward`/`.many` (kev/model.py)
    skip the temperature division entirely when `self.temperature == 1.0`: `z if ... or
    self.temperature == 1.0 else z / self.temperature`. So temporarily setting
    `model.head.temperature = 1.0` and calling `model.probs(enc)` returns `softmax(z)` -- the
    model's own T=1 distribution, at full float32 precision, with no rounding anywhere in the
    path (we never call kev.api.to_answers). The as-served distribution at the checkpoint's own
    temperature T is then recovered exactly (not approximately) from that single T=1 result via
    the power-transform identity in the module docstring: p_T = normalize(p_1 ** (1/T)). This
    means one forward pass gives both distributions at full precision, which both satisfies "do
    not run the whole benchmark twice" and improves on running the served endpoint once and a
    KEV_TEMPERATURE=1.0 server a second time (that alternative would still round both).

    CUDA graphs and the fused Qwen3.5 kernels are both declined (LoadOptions(cuda_graphs=False,
    fused=False)): kev.serve turns both on by default for throughput, but a captured CUDA graph
    bakes in whatever `model.head.temperature` was at *capture* time -- toggling the Python
    attribute between calls would silently stop affecting a replayed graph. Eager mode is exact
    and this backend is not batching (concurrency=1, one request at a time), so the throughput
    cost is the right trade for correctness.
    """

    concurrency = 1

    def __init__(self, tag):
        self.tag = tag
        self.repo = KEV_REPOS[tag]
        self._loaded = False

    async def __aenter__(self):
        if not self._loaded:
            await _run_sync(self._load)
            self._loaded = True
        return self

    async def __aexit__(self, *a):
        return None

    def _load(self):
        import torch
        from kev.checkpoint import Checkpoint, LoadOptions

        self.revision = _model_sha(self.repo)
        ck = Checkpoint(f"{self.repo}@{self.revision}")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.bfloat16 if device == "cuda" else torch.float32
        opts = LoadOptions(dtype=dtype, cuda_graphs=False, fused=False)
        self.tok, self.model = ck.load(device, opts)
        self.device = device
        self.served_temperature = float(self.model.head.temperature)

    async def answer(self, rec):
        from kev.api import SystemOneRequest, to_record
        from kev.model import SERVE_MAX_BRANCH, SERVE_MAX_STATE

        req = SystemOneRequest(state=rec["state"], questions=rec["questions"])
        internal_rec, meta = to_record(req)

        def _score():
            enc = self.model.encode(self.tok, internal_rec, max_state=SERVE_MAX_STATE,
                                    max_branch=SERVE_MAX_BRANCH)
            orig_T = self.model.head.temperature
            self.model.head.temperature = 1.0
            try:
                rows = self.model.probs(enc)
            finally:
                self.model.head.temperature = orig_T
            return rows, len(enc["ids"])

        rows, n_tok = await _run_sync(_score)
        T = self.served_temperature
        out = {}
        for row, m in zip(rows, meta):
            p1 = {k: float(v) for k, v in zip(m["keys"], row.tolist())}
            p_served = p1 if T == 1.0 else _pow_normalize(p1, 1.0 / T)
            raw = {"served_temperature": T, "probs_t1": p1,
                  "temperature_source": "kev.checkpoint.Meta.temperature (head.pt)"}
            if m["type"] == "score":
                raw["legend"] = m.get("legend")
            out[m["id"]] = {"probs": p_served, "raw": raw}
        return {"model_returned": self.tag, "usage_in": n_tok, "answers": out}


harness.register("kev-0.8b", lambda **kw: KevBackend("kev-0.8b", **kw))
harness.register("kev-9b", lambda **kw: KevBackend("kev-9b", **kw))


# ---------------------------------------------------------------------- decider-2b

DECIDER_REPO = "Mapika/decider-2b"


def isolated_score_t1(fit_served, T):
    """Given decider's per-level "fits" (P(this level fits), each an isolated yes/no row
    softmaxed at the served temperature T) and that same T, return the per-level fits a T=1
    forward pass would have produced. Each row is its own 2-option {no, yes} softmax, so the
    module's power-transform identity applies row by row: p_1(yes) = normalize(p_T ** T)[yes].
    Pure function (no decider import) so test_adapters.py can check it against hand-picked
    numbers; the caller still has to recombine the result with decider's own combine_isolated so
    the two code paths agree on everything except the fit values themselves."""
    return {k: _pow_normalize({"no": 1.0 - v, "yes": v}, T)["yes"] for k, v in fit_served.items()}


class DeciderBackend:
    """Mapika/decider-2b (v10), served at T=1.3 (decider_config.json's "temperature"), 
    "independent=True" (the default, and what we pass explicitly per the task).

    The bundled `decider/` package ships inside the model repo itself (see the model card's own
    usage snippet: "decider/ is included in this repo"), not on PyPI, so it is loaded by
    downloading a pinned snapshot and inserting that local directory onto sys.path -- this is also
    how the revision gets pinned, since decider.infer.Decider(path) takes a local path or an
    unpinned hub id, no revision kwarg.

    Precision. decider/systemone.py's `assemble()` calls `format_answer(rqs[k], probs[s])`, and
    `format_answer(rq, p, nd=4)` rounds every probability (and, for isolated Score levels, every
    per-level `level_fit`) to 4 decimals by default. `assemble` looks `format_answer` up as a bare
    module-global at call time, so reassigning `decider.systemone.format_answer` to
    `functools.partial(format_answer, nd=17)` (round to 17 decimals is a no-op for a float64/32
    value; there is no "don't round at all" switch) takes effect on every later `system_one()`
    call. `combine_isolated` and the certainty/confidence math are untouched.

    T=1 recovery. decider_config.json sets `isolated_levels: true`: a Score question's levels are
    each scored as an isolated yes/no row (its own softmax at T), and `combine_isolated` just
    ratio-normalizes those per-level "fit" values across levels -- that combination step is *not*
    itself a softmax, so the module's plain power-transform on the final Score distribution would
    not recover the correct T=1 Score distribution. `isolated_score_t1` above applies the
    power-transform to each isolated row first (which *is* exact, since each row is its own
    2-option softmax), then this backend recombines with decider's own `combine_isolated` so both
    the served and the T=1 Score distributions were produced by the identical combination logic.
    Choice and noul questions are plain softmaxes over their options, so the direct power-transform
    applies.
    """

    tag = "decider-2b"
    concurrency = 1

    def __init__(self):
        self._loaded = False

    async def __aenter__(self):
        if not self._loaded:
            await _run_sync(self._load)
            self._loaded = True
        return self

    async def __aexit__(self, *a):
        return None

    def _load(self):
        # decider/engine.py's Engine defaults to compile=True and decider/schema_engine.py
        # compiles per-schema graphs; both are skipped on CPU by passing use_graphs=False below,
        # but something else in the decider/transformers forward path still reaches
        # torch.compile's Inductor backend on a CPU-only machine with no Triton (observed:
        # "InductorError: AssertionError: Invalid device id"). These two env vars force
        # dynamo/inductor off process-wide. They must be set before `import torch` -- the very
        # first torch import in this process, since nothing before this point in harness.py or
        # adapters.py touches torch -- not merely before `import decider`: setting them after
        # `import torch` but before `import decider` was verified NOT to prevent the error, only
        # setting them before `import torch` itself did. They are set unconditionally, then
        # removed again if this turns out to be a CUDA machine (Colab), where decider's own
        # compile=True optimisations should run normally; on CUDA the corresponding branch below
        # never imports decider.* until after they are removed.
        _had = {v: v in os.environ for v in ("TORCHDYNAMO_DISABLE", "TORCH_COMPILE_DISABLE")}
        os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
        os.environ.setdefault("TORCH_COMPILE_DISABLE", "1")

        import sys
        import huggingface_hub as hh
        import torch

        # torch.compile and CUDA graphs stay off on every device (decision of the
        # coordinator): eager mode is slower but has no compile-time failure modes on an
        # unattended Colab run, and latency is reported with this setting stated.

        self.revision = _model_sha(DECIDER_REPO)
        local_dir = hh.snapshot_download(DECIDER_REPO, revision=self.revision)
        if local_dir not in sys.path:
            sys.path.insert(0, local_dir)

        import decider.systemone as _so
        if not getattr(_so, "_p4_unrounded", False):
            import functools
            _so.format_answer = functools.partial(_so.format_answer, nd=17)
            _so._p4_unrounded = True
        self._combine_isolated = _so.combine_isolated

        from decider.infer import Decider
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.bfloat16 if device == "cuda" else torch.float32
        self.decider = Decider(local_dir, device=device, dtype=dtype, use_graphs=False)
        self.served_temperature = float(self.decider.T)

    async def answer(self, rec):
        def _score():
            return self.decider.system_one(rec["state"], rec["questions"], independent=True)

        result = await _run_sync(_score)
        answers, T = result["answers"], self.served_temperature
        out = {}
        for qid, q in rec["questions"].items():
            a = answers[qid]
            if q["type"] == "score" and "level_fit" in a:
                fit_served = {k: float(v) for k, v in a["level_fit"].items()}
                fit_t1 = isolated_score_t1(fit_served, T)
                order = sorted(fit_t1, key=int)
                p1_list, _mass = self._combine_isolated([fit_t1[k] for k in order])
                p1 = dict(zip(order, p1_list))
                probs_served = {k: float(v) for k, v in a["probabilities"].items()}
            elif q["type"] == "noul":
                probs_served = {"false": 1.0 - float(a["noul"]), "true": float(a["noul"])}
                p1 = _pow_normalize(probs_served, T)
            else:
                probs_served = {k: float(v) for k, v in a["probabilities"].items()}
                p1 = _pow_normalize(probs_served, T)
            out[qid] = {"probs": probs_served,
                       "raw": {"served_temperature": T, "probs_t1": p1, "native": a}}
        return {"model_returned": result.get("model"),
               "usage_in": (result.get("usage") or {}).get("input_tokens"),
               "answers": out}


harness.register("decider-2b", lambda **kw: DeciderBackend(**kw))


# ---------------------------------------------------------------------- this-that-1.0

THISTHAT_REPO = "flock-io/this-that-model-1.0"
THISTHAT_GITHUB_COMMIT = "542d445efa5f68b14bfbd1f8ed25aacd8379d839"   # FLock-io/this-that-model@main, resolved 2026-09-24


def thisthat_render(rec):
    """(state_text, [(qid, instructions, option_strings, option_keys), ...]) under the fixed
    rendering the task specifies for this model (it has no wire-format entry point -- see the
    backend's docstring). Pure function; test_adapters.py exercises it without importing thisthat.
    """
    state_text = _state_text(rec["state"])
    items = []
    for qid, q in rec["questions"].items():
        t = q["type"]
        if t == "choice":
            keys = list(q["criteria"])
            opts = [f"{k}: {_txt(q['criteria'][k])}" for k in keys]
        elif t == "noul":
            keys, opts = ["false", "true"], ["no", "yes"]
        elif t == "score":
            crit = q["criteria"]
            keys = [str(i) for i in range(len(crit))]
            opts = [f"{i}: {_txt(c)}" for i, c in enumerate(crit)]
        else:
            raise ValueError(f"unknown question type {t!r}")
        items.append((qid, q["instructions"], opts, keys))
    return state_text, items


class ThisThatBackend:
    """flock-io/this-that-model-1.0.

    TypedDecider exposes no system_one / wire-format entry point: checked against
    thisthat/server.py at the pinned GitHub commit (THISTHAT_GITHUB_COMMIT above) -- it serves an
    OpenAI-style /v1/chat/completions with the option set given as a response_format enum, not
    TypeSafe's {state, questions} shape, and thisthat/model.py's TypedDecider only has
    decide()/decide_batch(). So this backend uses the task's prescribed fallback rendering:
      - choice option string: f"{key}: {description}"
      - noul: options ["no", "yes"], answers mapped back to keys "false"/"true"
      - score option string: f"{i}: {level description}"
      - state: as-is if already a string, else json.dumps(state, ensure_ascii=False)

    No calibration file ships in the repo (its listing has no temperature_config.json, unlike
    Nimble's), and decide()'s own default is temperature=1.0, so that default *is* the served
    temperature; served and T=1 are identical here. thisthat/model.py never rounds
    Decision.probabilities (built as `tuple(x / total for x in p)` from a float32 softmax, no
    round() call in the module), so no precision hook is needed.
    """

    tag = "this-that-1.0"
    concurrency = 1

    def __init__(self):
        self._loaded = False

    async def __aenter__(self):
        if not self._loaded:
            await _run_sync(self._load)
            self._loaded = True
        return self

    async def __aexit__(self, *a):
        return None

    def _load(self):
        import huggingface_hub as hh
        self.revision = _model_sha(THISTHAT_REPO)
        local_dir = hh.snapshot_download(THISTHAT_REPO, revision=self.revision)
        from thisthat import TypedDecider
        self.decider = TypedDecider.from_pretrained(local_dir)
        self.served_temperature = 1.0

    async def answer(self, rec):
        from thisthat import Question

        state_text, items = thisthat_render(rec)
        questions = [Question(instr, opts) for _, instr, opts, _ in items]

        def _score():
            # a list in, a list out (TypedDecider.decide only unwraps to a single Decision when
            # given a bare Question, never when given a list -- see thisthat/model.py:decide()),
            # so `decisions` is already aligned with `items` regardless of how many questions.
            return self.decider.decide(state_text, questions, temperature=1.0)

        decisions = await _run_sync(_score)
        out = {}
        for (qid, _instr, _opts, keys), d in zip(items, decisions):
            probs = {k: float(p) for k, p in zip(keys, d.probabilities)}
            out[qid] = {"probs": probs,
                       "raw": {"served_temperature": 1.0, "probs_t1": probs,
                               "choice": d.choice, "index": d.index}}
        return {"model_returned": self.tag, "usage_in": None, "answers": out}


harness.register("this-that-1.0", lambda **kw: ThisThatBackend(**kw))


# ---------------------------------------------------------------------- nimble-9b

NIMBLE_REPO = "bespokelabs/Bespoke-Nimble-9B"


def nimble_schema(questions):
    """(schema, score_fields) for inference.ParallelScorer.score(), per the task's mapping:
      noul   -> {"type": "boolean", "description": instructions}
      choice -> {"type": "enum", "choices": [keys], "description": instructions,
                 "choice_descriptions": {key: desc}}
      score  -> an integer-string enum ["0", ...] with level descriptions; the field name is
                listed in score_fields

    Field names checked against extended_schema.validate_schema (bespokelabs/Bespoke-Nimble-9B's
    own serving_schema.py, fetched at the pinned revision): "type" is "enum" or "boolean";
    "choices" is a list of distinct nonempty strings for "enum" (omitted for "boolean", which
    always means [False, True]); "choice_descriptions" is keyed by choice_key(value) -- "false"/
    "true" for a boolean field, the literal string for an enum field. Pure function, no
    torch/transformers import, so test_adapters.py exercises it directly.

    The question id is shown to the model as the schema's field name here (Nimble's
    prepare_prompts serialises {"name": name, "description": ..., "choices": [...]} verbatim into
    the prompt) -- unlike every other adapter in this module, where the qid never reaches the
    model. That is inherent to Nimble's one-JSON-object-keyed-by-field-name schema, so the freeze
    files' plain qids ("action", "risk", "intent", ...) are what this model reads; documented here
    since it is the one place that differs.
    """
    schema, score_fields = {}, []
    for qid, q in questions.items():
        t = q["type"]
        if t == "noul":
            schema[qid] = {"type": "boolean", "description": _txt(q["instructions"])}
        elif t == "choice":
            keys = list(q["criteria"])
            schema[qid] = {"type": "enum", "choices": keys, "description": _txt(q["instructions"]),
                           "choice_descriptions": {k: _txt(q["criteria"][k]) for k in keys}}
        elif t == "score":
            crit = q["criteria"]
            keys = [str(i) for i in range(len(crit))]
            schema[qid] = {"type": "enum", "choices": keys, "description": _txt(q["instructions"]),
                           "choice_descriptions": {str(i): _txt(c) for i, c in enumerate(crit)}}
            score_fields.append(qid)
        else:
            raise ValueError(f"unknown question type {t!r}")
    return schema, score_fields


class NimbleBackend:
    """bespokelabs/Bespoke-Nimble-9B, main branch: updated 2026-09-24 to a T=1.0 checkpoint (the
    task's "pin the main commit SHA as of today"). The pre-update checkpoint stays available at
    revision="original-2676" and is not used. LoRA adapter on Qwen/Qwen3.5-9B; the base repo and
    revision come from schema_config.json's own "model"/"revision" fields, which
    inference.ParallelScorer reads and loads itself -- parallel_schema.MODEL_ID
    ("Qwen/Qwen3.5-4B") is a stale constant left over from an earlier prompt-contract version and
    is NOT what actually gets loaded; schema_config.json says Qwen/Qwen3.5-9B @
    c202236235762e1c871ad0ccb60c8ee5ba337b9a, which matches the model card.

    Requires a CUDA GPU with bf16 support: inference.py hard-fails otherwise ("if not
    torch.cuda.is_available() or not torch.cuda.is_bf16_supported(): raise RuntimeError(...)"),
    with no fallback path. The project laptop's T1000 is Turing (no bf16), so this backend can
    only be import-tested there; the code is written to run unmodified on the Colab L4.

    Precision: inference.py's decision_result() never rounds -- probabilities and logits are
    plain `.tolist()` of a float64 tensor (`logits.double()`). Temperature is fixed at 1.0 for
    this checkpoint (ParallelScorer's default argument, and the model card's September 24 update
    says so explicitly; temperature_config.json in the repo is the *previous* checkpoint's 2.179
    and is not read by this code path), so served and T=1 are identical and no precision hook is
    needed.
    """

    tag = "nimble-9b"
    concurrency = 1

    def __init__(self):
        self._loaded = False

    async def __aenter__(self):
        if not self._loaded:
            await _run_sync(self._load)
            self._loaded = True
        return self

    async def __aexit__(self, *a):
        return None

    def _load(self):
        import sys
        import huggingface_hub as hh
        self.revision = _model_sha(NIMBLE_REPO)
        local_dir = hh.snapshot_download(NIMBLE_REPO, revision=self.revision)
        if local_dir not in sys.path:
            sys.path.insert(0, local_dir)
        from inference import NimbleModel   # = ParallelScorer; hard-requires CUDA + bf16
        self.model = NimbleModel(local_dir, temperature=1.0)
        self.served_temperature = 1.0

    async def answer(self, rec):
        schema, score_fields = nimble_schema(rec["questions"])
        context = _state_text(rec["state"])

        def _score():
            return self.model.score(context, schema, score_fields=score_fields)

        result = await _run_sync(_score)
        fields = result["fields"]
        out = {}
        for qid in rec["questions"]:
            f = fields[qid]
            probs = {k: float(v) for k, v in f["probabilities"].items()}
            out[qid] = {"probs": probs,
                       "raw": {"served_temperature": 1.0, "probs_t1": probs,
                               "logits": f.get("logits"), "native": f}}
        return {"model_returned": self.tag, "usage_in": None, "answers": out}


harness.register("nimble-9b", lambda **kw: NimbleBackend(**kw))


# ---------------------------------------------------------------------- comparator-open (vLLM)

# Qwen3-14B, official AWQ 4-bit release. The first Colab run used gemma-3-27b-it-int4-awq with
# max_model_len 16384 and failed at start-up on the L4 (2026-09-24); a 27B model leaves too little
# KV cache on 24 GB for batched generation. The 14B official AWQ checkpoint is natively supported
# by vLLM, leaves ample KV cache, and is run in non-thinking mode through its chat template.
COMPARATOR_REPO = "Qwen/Qwen3-14B-AWQ"
COMPARATOR_FULL_COVERAGE_MAX_K = 20   # every option reported at K <= this, else the top 5
COMPARATOR_TOP_N = 5


def comparator_option_list(q):
    """(keys, descriptions) in option order, using the same key convention as every other
    adapter in this module (wire_keys)."""
    t = q["type"]
    if t == "choice":
        keys = list(q["criteria"])
        descs = [_txt(q["criteria"][k]) for k in keys]
    elif t == "noul":
        c = noul_criteria(q)
        keys, descs = ["false", "true"], [_txt(c["false"]), _txt(c["true"])]
    else:
        crit = q["criteria"]
        keys = [str(i) for i in range(len(crit))]
        descs = [_txt(c) for c in crit]
    return keys, descs


def comparator_prompt(rec, qid, q):
    """(prompt_text, keys, n_report). n_report is how many {key, p} pairs the model must supply:
    every option at K <= 20, the 5 it considers most likely otherwise (the task's own rule)."""
    keys, descs = comparator_option_list(q)
    k = len(keys)
    n_report = k if k <= COMPARATOR_FULL_COVERAGE_MAX_K else COMPARATOR_TOP_N
    lines = "\n".join(f"- {key}: {desc}" for key, desc in zip(keys, descs))
    coverage = ("every option listed above" if k <= COMPARATOR_FULL_COVERAGE_MAX_K
               else f"the {COMPARATOR_TOP_N} options you judge most likely")
    prompt = (
        "You are answering a typed decision question about the state below. Reply with a single "
        "JSON object and nothing else.\n\n"
        f"State:\n{_state_text(rec['state'])}\n\n"
        f"Question: {_txt(q['instructions'])}\n\nOptions:\n{lines}\n\n"
        "Return JSON of the form "
        '{"answer": <the single best option\'s key, exactly as written above>, '
        '"probabilities": [{"key": <an option key>, "p": <your probability for it>}, ...]}, '
        f"covering {coverage}. Report your genuine calibrated belief for each; the probabilities "
        "need not sum to exactly 1, they will be renormalized."
    )
    return prompt, keys, n_report


def comparator_json_schema(n_report):
    """A guided-decoding JSON schema using an array of {key, p} pairs rather than an object keyed
    by the option strings themselves. An object whose property names are drawn from a dynamic,
    per-request enum is weakly supported by structured-output backends at K > ~20 and awkward to
    validate reliably at any K; the array form is uniform for every question regardless of option
    count and is converted back into the {"answer", "probabilities": {key: p}} shape the task
    describes by comparator_parse below."""
    return {
        "type": "object",
        "properties": {
            "answer": {"type": "string"},
            "probabilities": {
                "type": "array", "minItems": n_report, "maxItems": n_report,
                "items": {"type": "object",
                         "properties": {"key": {"type": "string"}, "p": {"type": "number"}},
                         "required": ["key", "p"]},
            },
        },
        "required": ["answer", "probabilities"],
    }


def comparator_parse(text, keys):
    """Parse the model's JSON reply into a full distribution over `keys`: missing options get 0,
    then the whole thing is renormalized (the task's own rule). Raises ValueError on a genuine
    parse failure, which the caller turns into an error line (the task: "parse failures as
    errors"). Pure function; test_adapters.py feeds it hand-written model output."""
    try:
        obj = json.loads(text)
        got = {}
        for item in obj["probabilities"]:
            k, p = str(item["key"]), float(item["p"])
            got[k] = max(0.0, p)
    except (json.JSONDecodeError, KeyError, TypeError, ValueError) as e:
        raise ValueError(f"comparator: could not parse the requested JSON: {e!r}") from e
    probs = {k: got.get(k, 0.0) for k in keys}
    s = sum(probs.values())
    if s <= 0:
        probs = {k: 1.0 / len(keys) for k in keys}   # every reported key missed the real option
                                                       # set: uniform, not an undefined 0/0 split
    else:
        probs = {k: v / s for k, v in probs.items()}
    return probs, obj.get("answer")


class ComparatorOpenBackend:
    """An open generative instruction model, served with vLLM offline batch inference on Colab:
    the "reads and reasons in text, then reports a number" comparator against the typed decision
    models. See COMPARATOR_REPO's comment above for which checkpoint and why.

    Batched through harness.py's documented prefetch() hook (this module's docstring): one vLLM
    LLM.generate() call per up-to-500-record chunk covers every question of every record in that
    chunk, each with its own per-question guided JSON schema. answer() then just looks its
    record's precomputed answers up in a cache prefetch() filled, so harness.py's own latency_s
    (timed around a single answer() call) reads near 0 for a prefetched record; the true cost is
    in raw instead: latency_mode="batched", batch_wall_s (that chunk's whole generate() wall time)
    and n_requests (per-question calls in that chunk) -- the task's own naming.

    A prompt that would not fit in max_model_len together with max_tokens of headroom is not sent
    to generate() at all (one over-length prompt would otherwise fail the whole batched call); it
    is recorded as a parse-failure-shaped error instead, same as a genuine JSON parse failure.

    Temperature 0 (greedy decoding -- distinct from the softmax "temperature" the other backends
    discuss): there is no output-token distribution to read a calibrated probability from, since
    the model reports its own numeric belief in text, so served_temperature is reported as 0.0 and
    there is no separate T=1 to recover (probs_t1 == probs).
    """

    tag = "comparator-open"
    repo = COMPARATOR_REPO
    served_temperature = 0.0
    concurrency = 500   # matches run()'s chunk size; prefetch() does the real batching

    def __init__(self, max_model_len=12288, max_tokens=768, gpu_memory_utilization=0.90):
        self.max_model_len = max_model_len
        self.max_tokens = max_tokens
        self.gpu_memory_utilization = gpu_memory_utilization
        self._loaded = False
        self._cache = {}

    async def __aenter__(self):
        if not self._loaded:
            await _run_sync(self._load)
            self._loaded = True
        return self

    async def __aexit__(self, *a):
        return None

    def _load(self):
        self.revision = _model_sha(self.repo)
        from vllm import LLM
        # quantization and dtype are read from the checkpoint's config (awq_marlin on an L4)
        self.llm = LLM(model=self.repo, revision=self.revision,
                       max_model_len=self.max_model_len,
                       gpu_memory_utilization=self.gpu_memory_utilization)
        self.tokenizer = self.llm.get_tokenizer()

    def _chat(self, prompt):
        """The instruct model's own chat template, with Qwen3's thinking mode switched off so
        the reply is the JSON object alone."""
        msgs = [{"role": "user", "content": prompt}]
        try:
            return self.tokenizer.apply_chat_template(msgs, tokenize=False,
                                                      add_generation_prompt=True,
                                                      enable_thinking=False)
        except TypeError:
            return self.tokenizer.apply_chat_template(msgs, tokenize=False,
                                                      add_generation_prompt=True)

    def _requests(self, rec):
        return [(rec["key"], qid) + comparator_prompt(rec, qid, q)
               for qid, q in rec["questions"].items()]

    async def prefetch(self, chunk):
        await _run_sync(self._prefetch_sync, chunk)

    def _prefetch_sync(self, chunk):
        import time
        from vllm import SamplingParams
        # structured output moved from guided_decoding=GuidedDecodingParams to
        # structured_outputs=StructuredOutputsParams in newer vLLM releases; support both
        try:
            from vllm.sampling_params import StructuredOutputsParams
        except ImportError:
            StructuredOutputsParams = None
        try:
            from vllm.sampling_params import GuidedDecodingParams
        except ImportError:
            GuidedDecodingParams = None

        all_reqs = [r for rec in chunk for r in self._requests(rec)]
        budget = self.max_model_len - self.max_tokens
        reqs, prompts, sampling = [], [], []
        for key, qid, prompt, keys, n_report in all_reqs:
            prompt = self._chat(prompt)
            n_prompt_tok = len(self.tokenizer.encode(prompt, add_special_tokens=False))
            if n_prompt_tok > budget:
                self._cache[(key, qid)] = {
                    "probs": None, "answer": None,
                    "error": f"comparator: prompt is {n_prompt_tok} tokens, budget is {budget}",
                    "raw_text": None, "usage_in": n_prompt_tok, "usage_out": None,
                    "latency_mode": "batched", "batch_wall_s": 0.0, "n_requests": len(all_reqs)}
                continue
            schema = comparator_json_schema(n_report)
            if StructuredOutputsParams is not None:
                sp = SamplingParams(temperature=0, max_tokens=self.max_tokens,
                                    structured_outputs=StructuredOutputsParams(json=schema))
            elif GuidedDecodingParams is not None:
                sp = SamplingParams(temperature=0, max_tokens=self.max_tokens,
                                    guided_decoding=GuidedDecodingParams(json=schema))
            else:   # older vLLM releases: guided_json lives directly on SamplingParams
                sp = SamplingParams(temperature=0, max_tokens=self.max_tokens, guided_json=schema)
            reqs.append((key, qid, keys))
            prompts.append(prompt)
            sampling.append(sp)

        if not prompts:
            return
        t0 = time.time()
        outputs = self.llm.generate(prompts, sampling)
        wall = time.time() - t0
        for (key, qid, keys), out in zip(reqs, outputs):
            text = out.outputs[0].text
            usage_in = len(out.prompt_token_ids) if out.prompt_token_ids is not None else None
            usage_out = len(out.outputs[0].token_ids) if out.outputs[0].token_ids is not None else None
            try:
                probs, answer = comparator_parse(text, keys)
                err = None
            except ValueError as e:
                probs, answer, err = None, None, str(e)
            self._cache[(key, qid)] = {
                "probs": probs, "answer": answer, "error": err, "raw_text": text,
                "usage_in": usage_in, "usage_out": usage_out,
                "latency_mode": "batched", "batch_wall_s": wall, "n_requests": len(reqs)}

    async def answer(self, rec):
        out, usage_in_total = {}, 0
        for qid in rec["questions"]:
            key = (rec["key"], qid)
            if key not in self._cache:
                await self.prefetch([rec])   # e.g. a direct unit test that never called prefetch
            cached = self._cache.pop(key)
            if cached["error"] is not None:
                raise RuntimeError(cached["error"])
            usage_in_total += cached["usage_in"] or 0
            out[qid] = {"probs": cached["probs"],
                       "raw": {"served_temperature": 0.0, "probs_t1": cached["probs"],
                               "answer": cached["answer"], "raw_text": cached["raw_text"],
                               "usage_out": cached["usage_out"], "latency_mode": cached["latency_mode"],
                               "batch_wall_s": cached["batch_wall_s"], "n_requests": cached["n_requests"]}}
        return {"model_returned": self.repo, "usage_in": usage_in_total, "answers": out}


harness.register("comparator-open", lambda **kw: ComparatorOpenBackend(**kw))

# Conditions the comparator answers, per the task and the coordinator's 2026-09-24 update (the
# refrozen manifest's D3 task list): d1_neutral, d1_calib, d2_k150, d2_k20, and every d3_<task> ,
# not the e2_ files.
COMPARATOR_CONDITIONS = ["d1_neutral", "d1_calib", "d2_k150", "d2_k20",
                        "d3_conv_go_awry", "d3_wiki_corpus", "d3_emotion", "d3_wiki_politeness"]


In [ ]:
%%writefile /content/p4/shared/src/s00b_prepare_d3.py
"""Rebuild shared/data/d3/*.jsonl and manifest.json from public sources.

Self-contained: fetches the two source repositories at pinned commits over
HTTP (raw.githubusercontent.com, no auth needed, both repos are public) and
writes the four D3 task files plus manifest.json. No local state is read;
everything needed is either a literal in this file or fetched over the
network. Intended to run unmodified on a fresh machine (e.g. Google Colab
on Linux) as well as on the original Windows workstation.

Sources
-------
- Question/option wording and label mapping: hazemibrahim97/decision-models-css
  commit 311956c2d1096cabc0f09a1f248e7dc0e1c0c41e, file pilot_jev.py
  (https://github.com/hazemibrahim97/decision-models-css). The relevant
  functions (`parse_prompt`, `task_criteria`, `OPTION_TO_GOLD`,
  `BINARY_CRITERIA`) are reproduced here by hand rather than imported,
  because importing pilot_jev.py reads an OpenRouter API key from disk at
  module load time, which this script must not do.
- Task text and gold labels: SALT-NLP/LLMs_for_CSS commit
  55183a64d7faaf6d5fc23eddf2bfc48ece4cacad, files
  css_data/<task>/test.json (https://github.com/SALT-NLP/LLMs_for_CSS).

Usage
-----
    python s00b_prepare_d3.py [--out DIR]

Writes <DIR>/<task>.jsonl for task in {conv_go_awry, wiki_corpus, emotion,
wiki_politeness} and <DIR>/manifest.json, with sha256 + n per file. Default
DIR is ../data/d3 relative to this file (i.e. shared/data/d3).

Dependencies: requests only (pandas/datasets/huggingface_hub are permitted
per the project's environment but are not needed for this script, since the
source data is plain JSON over HTTP, not a Hugging Face dataset).
"""

import argparse
import hashlib
import json
import random
import re
import pathlib

import requests

SEED = 20260924

SALT_REPO = "SALT-NLP/LLMs_for_CSS"
SALT_COMMIT = "55183a64d7faaf6d5fc23eddf2bfc48ece4cacad"
IBRAHIM_ZAKI_REPO = "hazemibrahim97/decision-models-css"
IBRAHIM_ZAKI_COMMIT = "311956c2d1096cabc0f09a1f248e7dc0e1c0c41e"  # provenance only; not fetched, logic is reproduced below

RAW_BASE = "https://raw.githubusercontent.com/{repo}/{commit}/{path}"

ALL_TASKS = ["conv_go_awry", "wiki_corpus", "emotion", "wiki_politeness"]
BINARY_TASKS = {"conv_go_awry", "wiki_corpus"}

# ---------------------------------------------------------------------
# Reproduced by hand from pilot_jev.py at IBRAHIM_ZAKI_COMMIT (see module
# docstring). Word for word except for the subset of OPTION_TO_GOLD /
# BINARY_CRITERIA entries this script needs.
# ---------------------------------------------------------------------

def parse_prompt(prompt):
    lines = prompt.strip().split("\n")
    instr, opts, cur = [], {}, None
    for line in lines:
        m = re.match(r"^([A-Z]+): (.*)$", line)
        if m:
            cur = m.group(1)
            opts[cur] = m.group(2)
        elif line.startswith("Constraint:"):
            cur = None
        elif cur:
            opts[cur] += " " + line.strip()
        else:
            instr.append(line)
    return " ".join(instr).strip(), opts


OPTION_TO_GOLD = {
    "emotion": {"A": "A", "B": "B", "C": "C", "D": "D", "E": "E", "F": "F"},
    "wiki_politeness": {"A": 1, "B": 0, "C": -1},
}

BINARY_CRITERIA = {
    "conv_go_awry": {
        "True": "True: the previous conversation eventually derails into a personal attack.",
        "False": "False: the previous conversation does not eventually derail into a personal attack.",
    },
    "wiki_corpus": {
        "True": "True: the named user is in a position of power in the conversation.",
        "False": "False: the named user is not in a position of power in the conversation.",
    },
}

# The marked/positive class per binary task: the outcome the task's own
# instruction text asserts. Both binary tasks here use the package's own
# "True" label for that marked outcome; "False" is the negation. See
# shared/data/d3/README.md for the full rationale.
POSITIVE_CLASS = {
    "conv_go_awry": "True",
    "wiki_corpus": "True",
}


def task_criteria(task, opts):
    if task in BINARY_CRITERIA:
        return BINARY_CRITERIA[task]
    criteria = {}
    for letter, gold in OPTION_TO_GOLD[task].items():
        text = opts.get(letter)
        criteria[str(gold)] = text.strip() if text else text
    return criteria


def coerce_gold(task, raw):
    if task in BINARY_TASKS:
        return str(raw)  # Python bool True/False -> "True"/"False"
    if task == "wiki_politeness":
        return str(raw)
    return raw  # emotion: already a letter string


# ---------------------------------------------------------------------
# Fetch
# ---------------------------------------------------------------------

def fetch_test_json(task):
    path = f"css_data/{task}/test.json"
    url = RAW_BASE.format(repo=SALT_REPO, commit=SALT_COMMIT, path=path)
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return json.loads(resp.text)


# ---------------------------------------------------------------------
# Build (identical logic to shared/src's scratch build used to produce the
# checked-in files; see shared/data/d3/README.md for the sampling note)
# ---------------------------------------------------------------------

def build_task(task):
    j = fetch_test_json(task)
    ids = sorted(j["labels"].keys())

    if task in BINARY_TASKS:
        options = ["True", "False"]
        criteria = task_criteria(task, {})
        per_item_instr = {}
        for pid in ids:
            instr, _opts = parse_prompt(j["prompts"][pid])
            per_item_instr[pid] = instr
    else:
        prompts = set(j["prompts"].values())
        assert len(prompts) == 1, (task, "prompt not fixed across items", len(prompts))
        prompt = next(iter(prompts))
        instr, opts = parse_prompt(prompt)
        criteria = task_criteria(task, opts)
        options = [str(g) for g in OPTION_TO_GOLD[task].values()]
        per_item_instr = {pid: instr for pid in ids}

    assert set(criteria.keys()) == set(options), (task, criteria.keys(), options)

    # Seed 20260924. Every task here has n <= 500 (Ziems et al.'s released
    # test.json is already a class-stratified sample of at most 500 items),
    # so this seeded draw selects every item; the seed fixes only a
    # deterministic per-label shuffle-then-concatenate write order.
    rnd = random.Random(SEED)
    by_label = {}
    for pid in ids:
        lbl = coerce_gold(task, j["labels"][pid])
        by_label.setdefault(lbl, []).append(pid)
    ordered = []
    for lbl in sorted(by_label.keys(), key=str):
        bucket = by_label[lbl][:]
        rnd.shuffle(bucket)
        ordered.extend(bucket[:500])
    rnd.shuffle(ordered)
    ordered = ordered[:500]

    rows = []
    for pid in ordered:
        text = j["context"][pid]
        gold = coerce_gold(task, j["labels"][pid])
        assert gold in criteria, (task, pid, gold, list(criteria.keys()))
        rows.append({
            "id": pid,
            "task": task,
            "text": text,
            "gold": gold,
            "options": options,
            "option_desc": criteria,
            "question": per_item_instr[pid],
            "binary": task in BINARY_TASKS,
            "positive": POSITIVE_CLASS.get(task) if task in BINARY_TASKS else None,
        })
    return rows


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=None, help="output directory (default: shared/data/d3 relative to this file)")
    args = ap.parse_args()

    out_dir = pathlib.Path(args.out) if args.out else (pathlib.Path(__file__).resolve().parent.parent / "data" / "d3")
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest = {}
    for task in ALL_TASKS:
        rows = build_task(task)
        path = out_dir / f"{task}.jsonl"
        write_jsonl(path, rows)
        digest = sha256_of(path)
        manifest[f"{task}.jsonl"] = {"sha256": digest, "n": len(rows)}
        print(task, "n=", len(rows), "sha256=", digest)

    manifest_path = out_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
    print("wrote", manifest_path)


if __name__ == "__main__":
    main()


## Embedded manifest

The checked-in `colab/inputs/manifest.json` this notebook must reproduce byte-for-byte per file (SHA-256), built at the same time as the source files above.

In [ ]:
EXPECTED_MANIFEST = json.loads('{\n "_retest_subset": {\n  "condition": "d1_neutral",\n  "item_ids": [\n   "agent_trace_observability_000011",\n   "agent_trace_observability_000020",\n   "agent_trace_observability_000028",\n   "agent_trace_observability_000048",\n   "agent_trace_observability_000054",\n   "agent_trace_observability_000057",\n   "agent_trace_observability_000058",\n   "agent_trace_observability_000072",\n   "agent_trace_observability_000077",\n   "agent_trace_observability_000094",\n   "customer_service_000019",\n   "customer_service_000027",\n   "customer_service_000031",\n   "customer_service_000045",\n   "customer_service_000046",\n   "customer_service_000064",\n   "customer_service_000075",\n   "customer_service_000084",\n   "customer_service_000090",\n   "customer_service_000096",\n   "invoice_processing_000000",\n   "invoice_processing_000002",\n   "invoice_processing_000006",\n   "invoice_processing_000019",\n   "invoice_processing_000020",\n   "invoice_processing_000024",\n   "invoice_processing_000032",\n   "invoice_processing_000047",\n   "invoice_processing_000055",\n   "invoice_processing_000090",\n   "security_incidents_000025",\n   "security_incidents_000032",\n   "security_incidents_000033",\n   "security_incidents_000038",\n   "security_incidents_000063",\n   "security_incidents_000065",\n   "security_incidents_000067",\n   "security_incidents_000071",\n   "security_incidents_000078",\n   "security_incidents_000079"\n  ]\n },\n "_seed": 20260924,\n "_sources": {\n  "d1": "https://huggingface.co/datasets/LocalLLaMA/typed-decisions revision c76749ec58bd8c3d2ea706b31c333a9059c38f90",\n  "d2": "https://github.com/clinc/oos-eval data/data_full.json and domains.json, commit 828f8093932c8fe6ca7936c3d2e52903b1c523de"\n },\n "d1_calib": {\n  "decisions": 1500,\n  "file": "d1_calib.jsonl",\n  "requests": 300,\n  "sha256": "e361865af906b14d76ef284a5bc59f66077e05b992783c435d649bbe74b3b93c"\n },\n "d1_native": {\n  "decisions": 2000,\n  "file": "d1_native.jsonl",\n  "requests": 400,\n  "sha256": "81614b52b78203cff75bf3824d0c2a10dc3a42588c7c97012c237d6a19952ba9"\n },\n "d1_neutral": {\n  "decisions": 2000,\n  "file": "d1_neutral.jsonl",\n  "requests": 400,\n  "sha256": "013fd3993a5d3e4e77cd267161d6333a202a35ca0daa2342abaf945f8548b591"\n },\n "d2_hier_dom": {\n  "decisions": 600,\n  "file": "d2_hier_dom.jsonl",\n  "requests": 600,\n  "sha256": "1179c57be59709f4dc9cd61719d6b805212834fd68d74196cdb342a40478f39d"\n },\n "d2_hier_int": {\n  "decisions": 600,\n  "file": "d2_hier_int.jsonl",\n  "requests": 600,\n  "sha256": "061bc29738de013edcfe9f7ccbbd978dc0687d963922c517b53ea5f8c65c0c4d"\n },\n "d2_k150": {\n  "decisions": 800,\n  "file": "d2_k150.jsonl",\n  "requests": 800,\n  "sha256": "b41706d09f2bd228463d50b5e1cd81d45c7da2fd3abaa5ce7fd869aaabd9a1ec"\n },\n "d2_k20": {\n  "decisions": 600,\n  "file": "d2_k20.jsonl",\n  "requests": 600,\n  "sha256": "d35c78e0a0541bc911268db6b56f4d644335ee76b1c907d4f9857a56b1041c98"\n },\n "d2_k5": {\n  "decisions": 600,\n  "file": "d2_k5.jsonl",\n  "requests": 600,\n  "sha256": "e0dbee0ded6b07ab9fd8b372287561c640f39ac436bb5782bcd24f6bea71d8a2"\n },\n "d2_k50": {\n  "decisions": 600,\n  "file": "d2_k50.jsonl",\n  "requests": 600,\n  "sha256": "7fd1cc8f33e947e01ca7136df96164ef09b7d5ea0ac65572a948df6ef9dfbb08"\n },\n "d3_conv_go_awry": {\n  "decisions": 500,\n  "file": "d3_conv_go_awry.jsonl",\n  "requests": 500,\n  "sha256": "de3ef7269832338cee4c271d23b9a4a18ea9919d55937bb1eb26612ee29fe712"\n },\n "d3_emotion": {\n  "decisions": 498,\n  "file": "d3_emotion.jsonl",\n  "requests": 498,\n  "sha256": "ccd5a620b9269d577a0f9672351b0eaa1b2f03fd0527eadaf9884e3078cc9f9b"\n },\n "d3_wiki_corpus": {\n  "decisions": 500,\n  "file": "d3_wiki_corpus.jsonl",\n  "requests": 500,\n  "sha256": "8b24c2f580b834d61971029eb4b8af94fc58d186b4673f3fe2a57b33557436d3"\n },\n "d3_wiki_politeness": {\n  "decisions": 498,\n  "file": "d3_wiki_politeness.jsonl",\n  "requests": 498,\n  "sha256": "a98b22a9efbac95da3dca46eaa40f5ac0b3f024527aa823f8e73446f74596e68"\n },\n "e2_d1_k01": {\n  "decisions": 600,\n  "file": "e2_d1_k01.jsonl",\n  "requests": 400,\n  "sha256": "9ea1cf5240cb73e4b795a10663afb687fc65228ef9cce5690ecb9d212fa0bf98"\n },\n "e2_d1_kny": {\n  "decisions": 600,\n  "file": "e2_d1_kny.jsonl",\n  "requests": 400,\n  "sha256": "73b0ebcfcfb734ac73f4c1f1e501b1f964344907d5114584c17b3878a79aa1f6"\n },\n "e2_d1_krand": {\n  "decisions": 600,\n  "file": "e2_d1_krand.jsonl",\n  "requests": 400,\n  "sha256": "4ae671a3e091c14e78dca097ea7cb590690ae898d7d54d5e1423f51af7401803"\n },\n "e2_d1_kswap": {\n  "decisions": 600,\n  "file": "e2_d1_kswap.jsonl",\n  "requests": 400,\n  "sha256": "106ab580a8274a88c9e504c688c8d0d1da232b3846f4d6ce55de6e55085c3ca8"\n },\n "e2_d3_conv_go_awry_k01": {\n  "decisions": 500,\n  "file": "e2_d3_conv_go_awry_k01.jsonl",\n  "requests": 500,\n  "sha256": "cf178f09d2a23b949b963cf527b1a8c55af6285196609edb3c79adbf8a02b728"\n },\n "e2_d3_conv_go_awry_kny": {\n  "decisions": 500,\n  "file": "e2_d3_conv_go_awry_kny.jsonl",\n  "requests": 500,\n  "sha256": "f3615a1592e3aad413ee0fa58d00d63635e9b2439d89aad2ed4755afd66c8c4f"\n },\n "e2_d3_conv_go_awry_krand": {\n  "decisions": 500,\n  "file": "e2_d3_conv_go_awry_krand.jsonl",\n  "requests": 500,\n  "sha256": "287bf4c6ba1e0c5af8f782869fa2bd5f01ba3a47de5613678801209b44e0802e"\n },\n "e2_d3_conv_go_awry_kswap": {\n  "decisions": 500,\n  "file": "e2_d3_conv_go_awry_kswap.jsonl",\n  "requests": 500,\n  "sha256": "329c2641487bf27954e5150fc05ec5652a7bcccd609402c6b4ee15a19b6e74a0"\n },\n "e2_d3_wiki_corpus_k01": {\n  "decisions": 500,\n  "file": "e2_d3_wiki_corpus_k01.jsonl",\n  "requests": 500,\n  "sha256": "a778a5c5904bb1234aaf256d593275c29ebfdc6460e25d0ad974b68dcb0d9bda"\n },\n "e2_d3_wiki_corpus_kny": {\n  "decisions": 500,\n  "file": "e2_d3_wiki_corpus_kny.jsonl",\n  "requests": 500,\n  "sha256": "01147e9e891551699ea9bcda03e890758e11aa16e7f90ea6a72b4409ae81eaa6"\n },\n "e2_d3_wiki_corpus_krand": {\n  "decisions": 500,\n  "file": "e2_d3_wiki_corpus_krand.jsonl",\n  "requests": 500,\n  "sha256": "f8ea689c90b1fe52e9748b93371fa28531c2bb5d1ec9f76a050601b2d063e9e6"\n },\n "e2_d3_wiki_corpus_kswap": {\n  "decisions": 500,\n  "file": "e2_d3_wiki_corpus_kswap.jsonl",\n  "requests": 500,\n  "sha256": "0c3248232c3fcb39aa3eb5ba32d91a09d2d587f6b9b43ae8a39ff6ff90656d22"\n }\n}')
print(len(EXPECTED_MANIFEST), 'entries in the embedded manifest')

## D3: rebuild the human-labelled social-science tasks

In [ ]:
%%bash
set -e
pip install -q requests
python3 /content/p4/shared/src/s00b_prepare_d3.py --out /content/p4/shared/data/d3


## D2: fetch CLINC-150 at the pinned commit

In [ ]:
import os, requests
os.makedirs('/content/p4/shared/data/d2', exist_ok=True)
CLINC_COMMIT = '828f8093932c8fe6ca7936c3d2e52903b1c523de'
for fn in ('data_full.json', 'domains.json'):
    url = f'https://raw.githubusercontent.com/clinc/oos-eval/{CLINC_COMMIT}/data/{fn}'
    r = requests.get(url, timeout=60); r.raise_for_status()
    with open(f'/content/p4/shared/data/d2/{fn}', 'wb') as fh:
        fh.write(r.content)
with open('/content/p4/shared/data/d2/CLINC_COMMIT', 'w') as fh:
    fh.write(CLINC_COMMIT + '\n')
print('fetched CLINC at', CLINC_COMMIT)

## Freeze the inputs

Determinism across machines: `datasets.load_dataset("LocalLLaMA/typed-decisions", ...)` is pinned to revision `c76749ec58bd8c3d2ea706b31c333a9059c38f90` (the same commit `s00_freeze_inputs.py`'s own `_sources` manifest entry already names) by a small monkeypatch below, not by editing s00_freeze_inputs.py itself -- that file also runs on the original workstation, where the unpinned call already happens to resolve that same revision (it is the dataset's current HEAD there), so pinning it only here keeps both call sites correct without duplicating the constant. Output files are written with `newline="\n"` (s00_freeze_inputs.py's own `write()`), so line endings cannot differ across OSes either.

In [ ]:
%%bash
set -e
pip install -q datasets huggingface_hub


In [ ]:
import sys, os, datasets as _hf_datasets
sys.path.insert(0, SRC_DIR)
os.chdir(SRC_DIR)
os.environ['HF_HOME'] = '/content/hf'

_orig_load_dataset = _hf_datasets.load_dataset
_D1_REPO, _D1_REVISION = 'LocalLLaMA/typed-decisions', 'c76749ec58bd8c3d2ea706b31c333a9059c38f90'
def _pinned_load_dataset(path, *a, **kw):
    if path == _D1_REPO and 'revision' not in kw:
        kw['revision'] = _D1_REVISION
    return _orig_load_dataset(path, *a, **kw)
_hf_datasets.load_dataset = _pinned_load_dataset

import s00_freeze_inputs
s00_freeze_inputs.main()

## Verify every frozen file against the embedded manifest

In [ ]:
import hashlib, json, os

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

with open(f'{INPUTS_DIR}/manifest.json', encoding='utf-8') as fh:
    got_manifest = json.load(fh)

mismatches = []
for cond, meta in EXPECTED_MANIFEST.items():
    if cond.startswith('_'):
        continue
    path = f'{INPUTS_DIR}/' + meta['file']
    if not os.path.exists(path):
        mismatches.append((cond, 'missing file', None, None))
        continue
    got = _sha256(path)
    if got != meta['sha256']:
        mismatches.append((cond, 'sha mismatch', meta['sha256'], got))

if mismatches:
    for cond, why, expect, got in mismatches:
        print(f'MISMATCH {cond}: {why} expected={expect} got={got}')
    raise SystemExit(f'{len(mismatches)} condition(s) did not reproduce the pinned inputs -- stopping.')
print(f'{len(EXPECTED_MANIFEST)} manifest entries verified against freshly frozen files.')

## Comparator

In [ ]:
import subprocess, sys, time, os, traceback
TAG = 'comparator-open'
PYBIN = '/content/envs/vllm_env/bin/python'
ANS_TAG_DIR = f'{ANSWERS_DIR}/{TAG}'
os.makedirs(ANS_TAG_DIR, exist_ok=True)
LOG = f'{ANS_TAG_DIR}/comparator_log.txt'
CONDS = ['d1_neutral', 'd1_calib', 'd2_k150', 'd2_k20',
         'd3_conv_go_awry', 'd3_wiki_corpus', 'd3_emotion', 'd3_wiki_politeness']

def stream(cmd, env=None):
    """Run a command, echoing every output line to the cell and appending it to the Drive log."""
    with open(LOG, 'a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(cmd)}\n'); log.flush()
        p = subprocess.Popen(cmd, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        return p.wait()

def harness_env():
    env = dict(os.environ)
    env['HF_HOME'] = '/content/hf'
    env['P4_INPUTS'] = INPUTS_DIR
    env['P4_ANSWERS'] = ANS_TAG_DIR
    env['VLLM_LOGGING_LEVEL'] = 'INFO'
    # FlashInfer JIT-compiles kernels with ninja, which lives in the env's bin folder;
    # greedy decoding does not need the FlashInfer sampler, so it is switched off as well
    env['PATH'] = '/content/envs/vllm_env/bin:' + env.get('PATH', '')
    env['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
    return env

if os.path.exists(f'{ANS_TAG_DIR}/ERROR.txt'):
    os.replace(f'{ANS_TAG_DIR}/ERROR.txt', f'{ANS_TAG_DIR}/ERROR_first_run.txt')
failed = []
try:
    print('=== comparator-open: building env ===', flush=True)
    rc = stream(['bash', '-c',
                 'set -e; export UV_CACHE_DIR=/content/uv-cache; '
                 'uv venv /content/envs/vllm_env --python 3.12 -q --allow-existing; '
                 'uv pip install --python /content/envs/vllm_env/bin/python -q '
                 'vllm huggingface_hub python-dotenv httpx numpy ninja; '
                 '/content/envs/vllm_env/bin/python -c "import vllm, torch; '
                 'print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"'])
    if rc:
        raise RuntimeError(f'env build failed (exit {rc})')
    print('=== comparator-open: smoke test, 5 requests of d2_k20 ===', flush=True)
    env = harness_env()
    env['P4_ANSWERS'] = '/content/smoke/comparator-open'
    rc = stream([PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', 'd2_k20',
                 '--rep', '1', '--limit', '5'], env=env)
    if rc:
        raise RuntimeError(f'smoke test failed (exit {rc}); see {LOG}')
    print('=== comparator-open: full run (one process, model loaded once) ===', flush=True)
    t0 = time.time()
    rc = stream([PYBIN, f'{SRC_DIR}/harness.py', '--model', TAG, '--cond', ','.join(CONDS),
                 '--rep', '1'], env=harness_env())
    print(f'full run exit {rc}, {(time.time() - t0) / 60:.1f} min', flush=True)
    if rc:
        raise RuntimeError(f'full run failed (exit {rc}); see {LOG}')
    with open(f'{ANS_TAG_DIR}/DONE', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    with open(f'{ANSWERS_DIR}/COMPARATOR_DONE', 'w') as fh:
        fh.write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()) + '\n')
    print('=== comparator-open: DONE ===')
except Exception:
    tb = traceback.format_exc()
    with open(f'{ANS_TAG_DIR}/ERROR.txt', 'w') as fh:
        fh.write(tb)
    print('=== comparator-open: ERROR; the full log is in ' + LOG + ' ===')
    print(tb)
